In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELDA 1: stable-diffusion.cpp CUDA en Kaggle (simple)
# ═══════════════════════════════════════════════════════════════

import os, sys, subprocess, shutil, json, time

        

FORCE_REBUILD_SDCPP = False

def sh(cmd, fatal=True):
    print(f"→ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (r.stderr or "") + (r.stdout or "")
    if r.returncode != 0:
        print(out[-5000:])
        if fatal:
            raise RuntimeError(f"Fallo: {cmd}")
    return r

IS_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or os.path.isdir("/kaggle/working")
if not IS_KAGGLE:
    raise RuntimeError("Esta celda está pensada para Kaggle.")

SDCPP_DIR = "/kaggle/working/stable-diffusion.cpp"
print(f"SDCPP_DIR: {SDCPP_DIR}")

result_np = subprocess.run(
    [sys.executable, "-c", "import numpy; print(numpy.__version__)"],
    capture_output=True, text=True
)
NUMPY_ORIGINAL = result_np.stdout.strip() or "2.0.2"
constraints_path = "/tmp/np_constraint.txt"
open(constraints_path, "w").write(f"numpy=={NUMPY_ORIGINAL}\n")
C = f"-c {constraints_path}"

sh("apt-get update -y > /dev/null 2>&1")
sh("apt-get install -y git cmake ninja-build ffmpeg > /dev/null 2>&1")
sh(f"pip install -q {C} openai edge-tts gradio psutil kaggle")
sh(f"pip install -q {C} faster-whisper ctranslate2>=4.5.0")
sh("pip install -q 'pyannote.audio>=4.0.1' torchcodec")
sh(f"pip install -q {C} pandas>=2.2.3 nltk omegaconf>=2.3.0 'huggingface-hub<1.0.0'")
sh("pip install -q --no-deps git+https://github.com/m-bain/whisperx.git")
sh(f"pip install -q {C} transformers==4.49.0 diffusers==0.32.0")
sh(f"pip install -q {C} langdetect")

if FORCE_REBUILD_SDCPP and os.path.exists(SDCPP_DIR):
    sh(f"rm -rf '{SDCPP_DIR}'")

if not os.path.exists(SDCPP_DIR):
    sh(f"git clone --recursive https://github.com/leejet/stable-diffusion.cpp '{SDCPP_DIR}'")
else:
    sh(f"git -C '{SDCPP_DIR}' reset --hard HEAD")
    # FIX: limpiar submódulos (evita el error de ggml con cambios locales)
    sh(f"git -C '{SDCPP_DIR}' submodule foreach --recursive 'git reset --hard || true'", fatal=False)
    sh(f"git -C '{SDCPP_DIR}' submodule foreach --recursive 'git clean -fdx || true'", fatal=False)

    # FIX PERSISTENCIA: no hacer pull si el binario ya existe (evita cambiar HEAD y forzar rebuild)
    _sdcli_existing = f"{SDCPP_DIR}/build/bin/sd-cli"
    _need_src_update = FORCE_REBUILD_SDCPP or (not os.path.exists(_sdcli_existing))

    if _need_src_update:
        sh(f"git -C '{SDCPP_DIR}' pull --recurse-submodules || true", fatal=False)
        sh(f"git -C '{SDCPP_DIR}' submodule update --init --recursive --force")
    else:
        # Solo asegurar submódulos en el commit ya fijado (sin mover HEAD)
        sh(f"git -C '{SDCPP_DIR}' submodule update --init --recursive || true", fatal=False)
        print("✓ Fuentes sd.cpp sin pull (binario existente). Se evita invalidar el build.")

r = sh("nvidia-smi -L", fatal=False)
if r.returncode != 0 or not (r.stdout or "").strip():
    raise RuntimeError("No hay GPU (nvidia-smi). Celda CUDA-only.")

r = sh("nvidia-smi --query-gpu=compute_cap --format=csv,noheader", fatal=False)
CUDA_ARCH = ((r.stdout or "").strip().splitlines() or ["7.5"])[0].replace(".", "") or "75"
print(f"CUDA_ARCH: {CUDA_ARCH}")

CUDA_HOME = "/usr/local/cuda"
NVCC = shutil.which("nvcc") or f"{CUDA_HOME}/bin/nvcc"
if not os.path.exists(NVCC):
    raise RuntimeError(f"nvcc no encontrado: {NVCC}")
sh(f"'{NVCC}' --version")

def find_libcuda_so1() -> str:
    r = subprocess.run("ldconfig -p 2>/dev/null | grep 'libcuda.so.1' || true",
                       shell=True, capture_output=True, text=True)
    for line in (r.stdout or "").splitlines():
        if "=>" in line:
            p = line.split("=>", 1)[1].strip()
            if os.path.exists(p):
                return p
    for p in [
        "/usr/local/nvidia/lib64/libcuda.so.1",
        "/usr/lib/x86_64-linux-gnu/libcuda.so.1",
        "/lib/x86_64-linux-gnu/libcuda.so.1",
    ]:
        if os.path.exists(p):
            return p
    r = subprocess.run(
        "find /usr/local/nvidia /usr/lib /lib -name 'libcuda.so.1' 2>/dev/null | head -n 1",
        shell=True, capture_output=True, text=True
    )
    p = (r.stdout or "").strip()
    return p if p and os.path.exists(p) else ""

LIBCUDA_SO1 = find_libcuda_so1()
print(f"libcuda.so.1: {LIBCUDA_SO1}")
if not LIBCUDA_SO1:
    raise RuntimeError("No se encontró libcuda.so.1")

CUDA_LINK_DIR = "/kaggle/working/cuda-link"
os.makedirs(CUDA_LINK_DIR, exist_ok=True)
LIBCUDA_SO = os.path.join(CUDA_LINK_DIR, "libcuda.so")

if os.path.lexists(LIBCUDA_SO):
    os.remove(LIBCUDA_SO)
os.symlink(LIBCUDA_SO1, LIBCUDA_SO)
sh(f"ls -l '{LIBCUDA_SO}'")

os.environ["LIBRARY_PATH"] = f"{CUDA_LINK_DIR}:/usr/local/nvidia/lib64:" + os.environ.get("LIBRARY_PATH", "")
os.environ["LD_LIBRARY_PATH"] = f"{CUDA_LINK_DIR}:/usr/local/nvidia/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["CPATH"] = f"{CUDA_HOME}/include:" + os.environ.get("CPATH", "")
os.environ["CUDA_PATH"] = CUDA_HOME
os.environ["CUDAToolkit_ROOT"] = CUDA_HOME

sdcli_bin = f"{SDCPP_DIR}/build/bin/sd-cli"
fp_path = f"{SDCPP_DIR}/build/BUILD_FINGERPRINT.json"

def git_head():
    r = subprocess.run(f"git -C '{SDCPP_DIR}' rev-parse HEAD", shell=True, capture_output=True, text=True)
    return (r.stdout or "").strip()

def up_to_date():
    if not (os.path.exists(sdcli_bin) and os.path.exists(fp_path)):
        return False
    try:
        fp = json.load(open(fp_path))
        return fp.get("commit") == git_head() and fp.get("backend") == "cuda" and fp.get("cuda_arch") == CUDA_ARCH
    except Exception:
        return False

if up_to_date():
    print("✓ sd-cli CUDA ya compilado. Se omite rebuild.")
else:
    sh(f"rm -rf '{SDCPP_DIR}/build'")

    cmake_cmd = (
        f"cd '{SDCPP_DIR}' && "
        f"cmake -S . -B build -G Ninja "
        f"-DCMAKE_BUILD_TYPE=Release "
        f"-DSD_CUDA=ON -DGGML_CUDA=ON "
        f"-DCMAKE_CUDA_COMPILER='{NVCC}' "
        f"-DCMAKE_CUDA_ARCHITECTURES={CUDA_ARCH} "
        f"-DCUDAToolkit_ROOT='{CUDA_HOME}' "
        f"-DCUDA_cuda_LIBRARY='{LIBCUDA_SO}' "
        f"-DCMAKE_LIBRARY_PATH='{CUDA_LINK_DIR};/usr/local/nvidia/lib64' "
        f"-DCMAKE_EXE_LINKER_FLAGS='-L{CUDA_LINK_DIR} -L/usr/local/nvidia/lib64'"
    )
    sh(cmake_cmd)
    sh(f"cd '{SDCPP_DIR}' && cmake --build build --config Release --target sd-cli")

    if not os.path.exists(sdcli_bin):
        raise RuntimeError("No se generó build/bin/sd-cli")

    os.makedirs(f"{SDCPP_DIR}/build", exist_ok=True)
    json.dump(
        {"commit": git_head(), "backend": "cuda", "cuda_arch": CUDA_ARCH, "libcuda": LIBCUDA_SO1},
        open(fp_path, "w"),
        indent=2,
    )
    print("¡Build CUDA exitoso!")

open(f"{SDCPP_DIR}/build/SDCLI_BACKEND.txt", "w").write("cuda")

# ---------------------------------------------------------------
# FIX: Kaggle persistence quita permisos de ejecución a binarios
# Restaurar +x en sd-cli y cualquier otro binario del build
# ---------------------------------------------------------------
if os.path.exists(sdcli_bin):
    os.chmod(sdcli_bin, 0o755)
    print(f"Permisos restaurados: {oct(os.stat(sdcli_bin).st_mode)}")

# También restaurar permisos en todos los binarios del directorio bin/
bin_dir = os.path.join(SDCPP_DIR, "build", "bin")
if os.path.isdir(bin_dir):
    for fname in os.listdir(bin_dir):
        fpath = os.path.join(bin_dir, fname)
        if os.path.isfile(fpath):
            os.chmod(fpath, 0o755)

print("\nVerificando sd-cli...")
r = subprocess.run([sdcli_bin, "--help"], capture_output=True, text=True)
help_txt = (r.stdout or "") + (r.stderr or "")
print(help_txt[:1000])
print(f"\nOK: {sdcli_bin}")
print(f"Qwen support: {'SÍ' if '--diffusion-model' in help_txt else 'NO'}")

print("\nINSTALACIÓN CUDA COMPLETADA. Reiniciando kernel...")
time.sleep(2)
os.kill(os.getpid(), 9)

SDCPP_DIR: /kaggle/working/stable-diffusion.cpp
→ apt-get update -y > /dev/null 2>&1
→ apt-get install -y git cmake ninja-build ffmpeg > /dev/null 2>&1
→ pip install -q -c /tmp/np_constraint.txt openai edge-tts gradio psutil kaggle
→ pip install -q -c /tmp/np_constraint.txt faster-whisper ctranslate2>=4.5.0
→ pip install -q 'pyannote.audio>=4.0.1' torchcodec
→ pip install -q -c /tmp/np_constraint.txt pandas>=2.2.3 nltk omegaconf>=2.3.0 'huggingface-hub<1.0.0'
→ pip install -q --no-deps git+https://github.com/m-bain/whisperx.git
→ pip install -q -c /tmp/np_constraint.txt transformers==4.49.0 diffusers==0.32.0
→ pip install -q -c /tmp/np_constraint.txt langdetect
→ git -C '/kaggle/working/stable-diffusion.cpp' reset --hard HEAD
→ git -C '/kaggle/working/stable-diffusion.cpp' submodule foreach --recursive 'git reset --hard || true'
→ git -C '/kaggle/working/stable-diffusion.cpp' submodule foreach --recursive 'git clean -fdx || true'
→ git -C '/kaggle/working/stable-diffusion.cpp' submodul

In [14]:
# ═══════════════════════════════════════════════════════════════
# CELDA 2: CONFIGURACIÓN GENERAL (Kaggle/Colab compatible) - MEJORADA
# ═══════════════════════════════════════════════════════════════

import os, gc, json, subprocess, shutil, time
from IPython.display import display, Javascript

IS_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or os.path.isdir("/kaggle/working")
IS_COLAB = False
try:
    import google.colab  # noqa: F401
    IS_COLAB = True
except Exception:
    IS_COLAB = False

KAGGLE_INPUT_DIR = "/kaggle/input"

if IS_KAGGLE:
    PLATFORM = "kaggle"
    OUTPUT_DIR = "/kaggle/working/VideoAutomationProject"
    SDCPP_DIR = "/kaggle/working/stable-diffusion.cpp"
elif IS_COLAB:
    PLATFORM = "colab"
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        OUTPUT_DIR = "/content/drive/MyDrive/VideoAutomationProject"
    except Exception as e:
        print(f"Drive no disponible ({e}). Usando almacenamiento local temporal.")
        OUTPUT_DIR = "/content/VideoAutomationProject"
    SDCPP_DIR = "/content/stable-diffusion.cpp"
else:
    PLATFORM = "generic"
    OUTPUT_DIR = os.path.abspath("./VideoAutomationProject")
    SDCPP_DIR = os.path.abspath("./stable-diffusion.cpp")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/images", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/audio_chunks", exist_ok=True)

CHECKPOINT_PATH = f"{OUTPUT_DIR}/checkpoint.json"
PROJECT_STATE_PATH = f"{OUTPUT_DIR}/project_state.json"

LOG_FILE = f"{OUTPUT_DIR}/run.log"
os.environ["PROJECT_LOG_FILE"] = LOG_FILE
if not os.path.exists(LOG_FILE):
    with open(LOG_FILE, "w", encoding="utf-8") as f:
        f.write("=== VideoAutomationProject LOG ===\n")

def _append_log(line: str):
    try:
        with open(LOG_FILE, "a", encoding="utf-8") as f:
            f.write(line.rstrip() + "\n")
    except Exception:
        pass

def _log_cell2(msg: str):
    ts = time.strftime("%H:%M:%S")
    line = f"[{ts}] {msg}"
    print(line)
    _append_log(line)

_log_cell2(f"Plataforma detectada: {PLATFORM}")
_log_cell2(f"OUTPUT_DIR: {OUTPUT_DIR}")
_log_cell2(f"SDCPP_DIR: {SDCPP_DIR}")
_log_cell2(f"LOG_FILE: {LOG_FILE}")

if IS_COLAB:
    js_keepalive = '''
    function KeepAlive() {
        const btn = document.querySelector("#top-toolbar > colab-connect-button");
        if (btn) btn.click();
    }
    setInterval(KeepAlive, 60000);
    console.log("Anti-desconexión activo");
    '''
    display(Javascript(js_keepalive))
    _log_cell2("Anti-desconexión (Colab) inyectado.")
else:
    _log_cell2("Anti-desconexión: omitido (no Colab).")

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

CPU_THREADS = max(1, (os.cpu_count() or 2) // 2)
os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS)

SDCLI_THREADS = max(1, os.cpu_count() or 2)

SDCLI_GPU_AVAILABLE = shutil.which("nvidia-smi") is not None
if SDCLI_GPU_AVAILABLE:
    try:
        r = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True, timeout=10)
        if r.returncode != 0 or not (r.stdout or "").strip():
            SDCLI_GPU_AVAILABLE = False
    except Exception:
        SDCLI_GPU_AVAILABLE = False

# ---------------------------------------------------------------
# API KEY xKIRO (HARDCODED)
# ⚠️ REEMPLAZA ESTO CON TU API KEY REAL
# ---------------------------------------------------------------
#XKIRO_API_KEY = "sk-xt-bbb1244e0de2a89bed4c5a7795803e74ddf0529154af5ee6"
#XKIRO_API_KEY =  "sk-xt-2f17ca00174697a0af0b693d40713fec96007fe0b3ad2de8"
XKIRO_API_KEY =  "sk-xt-4a5373fc29fc7292c082ad27e10b5333952cacf3fbf30c1c"
XKIRO_BASE_URL = "https://api.xkiro.com/v1"
os.environ["XKIRO_API_KEY"] = XKIRO_API_KEY

XKIRO_MODEL_OPTIONS = {
    "🇫🇷 Mistral Large 3 (MoE, el más capaz)":            "mistralai/mistral-large-2512",
    "🇫🇷 Mistral Medium 3.5 (agentic, texto+imagen)":     "mistralai/mistral-medium-3.5",
    "🇫🇷 Mistral Small 4 (instruct+razonamiento+código)": "mistralai/mistral-small-2603",
    "🇫🇷 Codestral (código)":                             "mistralai/codestral-2508",
    "🇫🇷 Devstral 2 Medium (agentic, código)":            "mistralai/devstral-medium",
    "🇫🇷 Ministral 3 14B (texto+visión)":                 "mistralai/ministral-14b",
    "🇫🇷 Ministral 3 8B (eficiente)":                     "mistralai/ministral-8b",
    "🇫🇷 Ministral 3 3B (ultraligero)":                   "mistralai/ministral-3b",
    "🌀 MiniMax M3 (1M contexto, frontier)":               "minimax/minimax-m3:free",
    "🌀 MiniMax M2.7":                                      "minimax/minimax-m2.7:free",
    "🌀 MiniMax M2.7 Highspeed":                            "minimax/minimax-m2.7-highspeed:free",
    "🌀 MiniMax M2.5 (código)":                             "minimax/minimax-m2.5:free",
    "🌀 MiniMax M2.5 Highspeed":                            "minimax/minimax-m2.5-highspeed:free",
    "🌀 MiniMax M2.1 (razonamiento)":                       "minimax/minimax-m2.1:free",
    "🌀 MiniMax M2.1 Highspeed":                            "minimax/minimax-m2.1-highspeed:free",
    "🌀 MiniMax M2 (agentic, function calling)":            "minimax/minimax-m2:free",
    "🟣 Qwen3.8 Omni Flash (multimodal, 1M)":               "qwen/qwen3.8-omni-flash:free",
    "🟣 Qwen3.8 Max (flagship, 1M) [RECOMENDADO]":          "qwen/qwen3.8-max:free",
    "🟣 Qwen3.7 Max":                                       "qwen/qwen3.7-max:free",
    "🟣 Qwen3.7 Plus (económico)":                          "qwen/qwen3.7-plus:free",
    "🟣 Qwen3.7 Flash (vision-language)":                   "qwen/qwen3.7-flash:free",
    "🟣 Qwen3.6 Plus (MoE híbrido)":                        "qwen/qwen3.6-plus:free",
    "🟣 Qwen3.6 Max Preview (~1T params)":                  "qwen/qwen3.6-max-preview:free",
    "🟣 Qwen3.6 27B (denso)":                               "qwen/qwen3.6-27b:free",
    "🟣 Qwen3.5 Plus":                                      "qwen/qwen3.5-plus:free",
    "🟣 Qwen3.5 Omni Plus (multimodal)":                    "qwen/qwen3.5-omni-plus:free",
    "🟣 Qwen3.6 35B A3B (multimodal)":                      "qwen/qwen3.6-35b-a3b:free",
    "🟣 Qwen3.5 Flash":                                     "qwen/qwen3.5-flash:free",
    "🟣 Qwen3.5 397B A17B (vision-language)":               "qwen/qwen3.5-397b-a17b:free",
    "🟣 Qwen3.5 Omni Flash":                                "qwen/qwen3.5-omni-flash:free",
    "🟣 Qwen3 Max":                                         "qwen/qwen3-max:free",
    "🟣 Qwen Plus 0728 (razonamiento híbrido)":             "qwen/qwen-plus-2025-07-28:free",
    "🟣 Qwen3 Coder Plus (código)":                         "qwen/qwen3-coder-plus:free",
    "🟣 Qwen3 VL Plus (visión, OCR)":                       "qwen/qwen3-vl-plus:free",
    "🟣 Qwen3 Omni Flash":                                  "qwen/qwen3-omni-flash:free",
    "🔵 SenseNova 6.8 Flash-Lite (multimodal)":             "sensenova/sensenova-6.8-flash-lite",
    "🔵 SenseNova 6.7 Flash-Lite (multimodal)":             "sensenova/sensenova-6.7-flash-lite",
}
DEFAULT_XKIRO_TEXT_MODEL_LABEL = "🟣 Qwen3.8 Max (flagship, 1M) [RECOMENDADO]"
DEFAULT_XKIRO_TEXT_MODEL_ID = XKIRO_MODEL_OPTIONS[DEFAULT_XKIRO_TEXT_MODEL_LABEL]
XKIRO_FALLBACK_POOL = [
    "qwen/qwen3.8-max:free",
    "minimax/minimax-m2:free",
    "mistralai/mistral-large-2512",
]

POLLINATIONS_MODELS = {
    "⚡ Flux (rápido, gratis e ilimitado) [RECOMENDADO]":       "flux",
    "🚀 Turbo (SDXL Turbo, ultra rápido)":                      "turbo",
    "🖼️ GPT Image (buena adherencia al prompt)":               "gptimage",
    "🎨 Kontext (bueno para composición)":                      "kontext",
    "🌅 Seedream (estético, buenos colores)":                   "seedream",
    "🔷 Z-Image (alternativa ligera)":                          "zimage",
}
DEFAULT_POLLINATIONS_MODEL = "⚡ Flux (rápido, gratis e ilimitado) [RECOMENDADO]"
POLLINATIONS_FALLBACK_MODEL = "flux"

IMAGE_BACKENDS = {
    "Pollinations (API gratis, rápido)": "pollinations",
    "Qwen-Image-2.1 Local (GGUF, sd-cli) [GPU rápido / CPU lento, según preset]": "qwen_local",
}
DEFAULT_IMAGE_BACKEND_LABEL = "Qwen-Image-2.1 Local (GGUF, sd-cli) [GPU rápido / CPU lento, según preset]"

SDCLI_PATH = f"{SDCPP_DIR}/build/bin/sd-cli"
try:
    with open(f"{SDCPP_DIR}/build/SDCLI_BACKEND.txt") as _f:
        SDCLI_BACKEND = _f.read().strip() or "cpu"
except Exception:
    SDCLI_BACKEND = "cpu"

LOCAL_MODELS_DIR = f"{OUTPUT_DIR}/local_models"
os.makedirs(LOCAL_MODELS_DIR, exist_ok=True)

LOCAL_MODELS_RUNTIME_DIR = "/content/qwen_models_cache"
if PLATFORM == "colab":
    os.makedirs(LOCAL_MODELS_RUNTIME_DIR, exist_ok=True)

QWEN21_VAE_URL  = "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/vae/qwen_image_2.1_vae_bf16.safetensors"
QWEN21_VAE_FILE = "qwen_image_2.1_vae_bf16.safetensors"

QWEN21_DIFFUSION_QUANTS = [
    ("Q2_K", "qwen_image_2.1-Q2_K.gguf", 2.56, True),
    ("Q3_K", "qwen_image_2.1-Q3_K.gguf", 3.27, True),
    ("Q4_0", "qwen_image_2.1-Q4_0.gguf", 4.20, True),
    ("Q4_K", "qwen_image_2.1-Q4_K.gguf", 4.20, True),
    ("Q5_0", "qwen_image_2.1-Q5_0.gguf", 5.07, True),
    ("Q6_K", "qwen_image_2.1-Q6_K.gguf", 6.00, True),
    ("Q8_0", "qwen_image_2.1-Q8_0.gguf", 7.69, False),
]
QWEN21_DIFFUSION_BASE_URL = "https://huggingface.co/leejet/Qwen-Image-2.1-GGUF/resolve/main"

QWEN21_ENCODER_QUANTS = [
    ("Q2_K", "https://huggingface.co/bartowski/Qwen_Qwen3-VL-8B-Instruct-GGUF/resolve/main/Qwen_Qwen3-VL-8B-Instruct-Q2_K.gguf",
     "Qwen_Qwen3-VL-8B-Instruct-Q2_K.gguf", 3.06, "comunidad/bartowski"),
    ("Q4_K_M", "https://huggingface.co/Qwen/Qwen3-VL-8B-Instruct-GGUF/resolve/main/Qwen3VL-8B-Instruct-Q4_K_M.gguf",
     "Qwen3VL-8B-Instruct-Q4_K_M.gguf", 5.03, "oficial/Qwen"),
]

QWEN21_LOCAL_PRESETS = {}
for _d_quant, _d_file, _d_gb, _d_confirmed in QWEN21_DIFFUSION_QUANTS:
    for _e_quant, _e_url, _e_file, _e_gb, _e_source in QWEN21_ENCODER_QUANTS:
        _total_gb = _d_gb + _e_gb + 0.64
        _confirm_tag = "" if _d_confirmed else " [DIFUSIÓN NO CONFIRMADA EN LISTADO — puede fallar la descarga]"
        _label = (
            f"Diffusion {_d_quant} + Encoder {_e_quant} "
            f"(~{_total_gb:.1f}GB total, encoder {_e_source}){_confirm_tag}"
        )
        QWEN21_LOCAL_PRESETS[_label] = {
            "diffusion_url": f"{QWEN21_DIFFUSION_BASE_URL}/{_d_file}",
            "diffusion_file": _d_file,
            "encoder_url": _e_url,
            "encoder_file": _e_file,
            "steps": 15,
            "cfg": 5.0,
            "cpu_lowram_params_backend": "diffusion=disk",
            "enable_mmap": True,
            "enable_vae_tiling": True,
            "offload_to_cpu_on_gpu": False,
        }

DEFAULT_QWEN21_PRESET_LABEL = next(
    lbl for lbl in QWEN21_LOCAL_PRESETS
    if lbl.startswith("Diffusion Q8_0 + Encoder Q4_K_M")
)

def _download_if_missing(url: str, out_path: str) -> None:
    if os.path.exists(out_path) and os.path.getsize(out_path) > 1024 * 1024:
        _log_cell2(f"Ya existe: {out_path} ({os.path.getsize(out_path)/1024**3:.2f} GB)")
        return
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    tmp_path = out_path + ".part"
    _log_cell2(f"Descargando:\n  {url}\n→ {out_path}")
    cmd = f"wget -c --progress=dot:giga -O '{tmp_path}' '{url}'"
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0 or (not os.path.exists(tmp_path)) or os.path.getsize(tmp_path) < 1024 * 1024:
        raise RuntimeError(f"Fallo descargando {url}")
    os.replace(tmp_path, out_path)
    _log_cell2(f"Descarga completada: {out_path} ({os.path.getsize(out_path)/1024**3:.2f} GB)")

def _find_in_kaggle_input(filename: str) -> str:
    if not os.path.isdir(KAGGLE_INPUT_DIR):
        return ""
    for root, _, files in os.walk(KAGGLE_INPUT_DIR):
        if filename in files:
            full = os.path.join(root, filename)
            if os.path.getsize(full) > 1024 * 1024:
                return full
    return ""

def _ensure_runtime_copy(persistent_path: str) -> str:
    if PLATFORM != "colab":
        return persistent_path
    if not persistent_path.startswith("/content/drive"):
        return persistent_path
    dst = os.path.join(LOCAL_MODELS_RUNTIME_DIR, os.path.basename(persistent_path))
    if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(persistent_path):
        _log_cell2(f"Copia local ya lista: {dst}")
        return dst
    _log_cell2(f"Copiando a disco local (más rápido que Drive): {persistent_path} → {dst}")
    shutil.copyfile(persistent_path, dst + ".part")
    os.replace(dst + ".part", dst)
    _log_cell2(f"Copia local completada: {dst}")
    return dst

def ensure_qwen21_local_files(preset_label: str) -> dict:
    preset = QWEN21_LOCAL_PRESETS[preset_label]

    def _resolve(filename, url):
        found = _find_in_kaggle_input(filename)
        if found:
            _log_cell2(f"Modelo encontrado en Dataset de Kaggle adjunto (SIN descargar): {found}")
            return found
        working_path = os.path.join(LOCAL_MODELS_DIR, filename)
        if os.path.exists(working_path) and os.path.getsize(working_path) > 1024 * 1024:
            _log_cell2(f"Modelo ya en caché persistente: {working_path}")
            return working_path
        _download_if_missing(url, working_path)
        return working_path

    diffusion_path = _resolve(preset["diffusion_file"], preset["diffusion_url"])
    llm_path       = _resolve(preset["encoder_file"], preset["encoder_url"])
    vae_path       = _resolve(QWEN21_VAE_FILE, QWEN21_VAE_URL)

    diffusion_path = _ensure_runtime_copy(diffusion_path)
    llm_path       = _ensure_runtime_copy(llm_path)
    vae_path       = _ensure_runtime_copy(vae_path)

    return {
        "diffusion_path": diffusion_path,
        "llm_path": llm_path,
        "vae_path": vae_path,
        "steps": int(preset["steps"]),
        "cfg": float(preset["cfg"]),
        "cpu_lowram_params_backend": preset.get("cpu_lowram_params_backend", None),
        "enable_mmap": bool(preset.get("enable_mmap", True)),
        "enable_vae_tiling": bool(preset.get("enable_vae_tiling", True)),
        "offload_to_cpu_on_gpu": bool(preset.get("offload_to_cpu_on_gpu", False)),
    }

def publish_models_as_kaggle_dataset(dataset_slug: str, title: str = None, update_existing: bool = True) -> str:
    if PLATFORM != "kaggle":
        return "Esta función solo aplica dentro de Kaggle."

    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        os.environ["KAGGLE_USERNAME"] = secrets.get_secret("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = secrets.get_secret("KAGGLE_KEY")
    except Exception as e:
        return (f"No se pudieron leer credenciales desde Kaggle Secrets ({e}). "
                f"Ve a Add-ons -> Secrets y agrega KAGGLE_USERNAME y KAGGLE_KEY.")

    staging_dir = f"{OUTPUT_DIR}/dataset_publish_staging"
    os.makedirs(staging_dir, exist_ok=True)
    for fname in os.listdir(LOCAL_MODELS_DIR):
        src = os.path.join(LOCAL_MODELS_DIR, fname)
        dst = os.path.join(staging_dir, fname)
        if os.path.isfile(src) and not os.path.exists(dst):
            shutil.copyfile(src, dst)

    metadata = {
        "title": title or dataset_slug,
        "id": f"{os.environ.get('KAGGLE_USERNAME', 'usuario')}/{dataset_slug}",
        "licenses": [{"name": "CC0-1.0"}],
    }
    with open(os.path.join(staging_dir, "dataset-metadata.json"), "w") as f:
        json.dump(metadata, f, indent=2)

    if update_existing:
        cmd = f"kaggle datasets version -p '{staging_dir}' -m 'Actualización de modelos Qwen-Image-2.1' -r zip"
    else:
        cmd = f"kaggle datasets create -p '{staging_dir}' -r zip"

    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode == 0:
        return f"Dataset publicado/actualizado correctamente: {metadata['id']}\n{r.stdout}"
    return f"Fallo al publicar el dataset:\n{r.stdout}\n{r.stderr}"

STYLE_GUIDE = {
    "art_style": (
        "Detailed 2D cartoon animation style, simple flat textures, thick and well-defined black lines with hand-drawn marker strokes, "
        "featuring rich and detailed background elements that naturally correspond to and enhance the main scene."
    ),
    "negative_prompt": (
        " no photorealism,  "
    ),
    "aspect_ratio": "16:9",
    "characters": (
        "Simple stick figures, large circular heads, dot eyes, expressive thick eyebrows. "

    ),
    "objects": "Simplified, robust cartoon shapes, single flat color fills, bold black outlines",
    "backgrounds": (
        "Solid flat color blocks only. White (default), green stripe (ground), "
        "blue sky + green ground (outdoor), orange (sunset/fire), "
        "beige/dark blue (ancient), solid blue (underwater/science)"
    ),
    "text_rules": (
        "Handwritten marker font, BOLD UPPERCASE, top of frame. "
        "Colors: RED, BLACK, or YELLOW ONLY, no shadows."
    ),
    "labels_arrows": "Black or yellow diagonal arrows pointing to objects, short UPPERCASE words next to the tip",
    "thought_bubbles": (
        "Classic cloud shape with UPPERCASE text inside "
        "(e.g., '¿HMMMM?', '?', '¡ESPERA...!', '¿QUÉ?', 'NO PUEDE SER...')"
    ),
    "color_palette": "#F5820D, #2D5FBF, #3A9E3A, #F5C518, #D94040, #8B5E3C, #6EB5E8, #C4965A, #FFFFFF",
    "framing_types": [
        "Conceptual text frame: Large centered object + BOLD UPPERCASE text at top",
        "Evolution sequence: Left-to-right progression with arrow",
        "Labeled diagram: Object with yellow diagonal arrow + UPPERCASE label",
        "Stick figure reaction: Thought bubble with '?', 'HMMMM', '!', 'ESPERA...'",
        "Personified villain: Abstract concept with angry cartoon face",
        "Globe + creatures: Centered globe surrounded by floating animals/objects",
    ],
}

VOICE_OPTIONS = {
    # =========================
    # AUTO (recomendado)
    # =========================
    "AUTO (según idioma del título)": "AUTO",

    # =========================
    # ESPAÑOL
    # =========================
    "Álvaro (Hombre, España)": "es-ES-AlvaroNeural",
    "Elvira (Mujer, España)":  "es-ES-ElviraNeural",
    "Ximena (Mujer, España)":  "es-ES-XimenaNeural",

    "Jorge (Hombre, México)":  "es-MX-JorgeNeural",
    "Dalia (Mujer, México)":   "es-MX-DaliaNeural",

    "Tomás (Hombre, Argentina)": "es-AR-TomasNeural",

    # (Si quieres más países, lo ideal es listarlas dinámicamente con edge_tts --list-voices)

    # =========================
    # ENGLISH
    # =========================
    "Jenny (Female, US)": "en-US-JennyNeural",
    "Guy (Male, US)":     "en-US-GuyNeural",
    "Aria (Female, US)":  "en-US-AriaNeural",

    "Sonia (Female, UK)": "en-GB-SoniaNeural",
    "Ryan (Male, UK)":    "en-GB-RyanNeural",

    # =========================
    # FRANÇAIS
    # =========================
    "Denise (Femme, France)": "fr-FR-DeniseNeural",
    "Henri (Homme, France)":  "fr-FR-HenriNeural",

    # =========================
    # PORTUGUÊS
    # =========================
    "Francisca (Mulher, Brasil)": "pt-BR-FranciscaNeural",
    "Antonio (Homem, Brasil)":    "pt-BR-AntonioNeural",
    "Raquel (Mulher, Portugal)":  "pt-PT-RaquelNeural",

    # =========================
    # 한국어 (KOREAN)
    # =========================
    "SunHi (Female, Korea)":  "ko-KR-SunHiNeural",
    "InJoon (Male, Korea)":   "ko-KR-InJoonNeural",

    # =========================
    # العربية (ARABIC)
    # Nota: “árabe más hablado” por hablantes nativos suele referirse a dialectos
    # (p.ej. egipcio). En Edge TTS hay ar-EG (Egipto) y ar-SA (Arabia Saudí).
    # =========================
    "Salma (Female, Arabic Egypt)": "ar-EG-SalmaNeural",
    "Shakir (Male, Arabic Egypt)":  "ar-EG-ShakirNeural",
    "Zariyah (Female, Arabic Saudi)": "ar-SA-ZariyahNeural",
    "Hamed (Male, Arabic Saudi)":     "ar-SA-HamedNeural",
}
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0  # determinista

# Idiomas objetivo del proyecto
LANG_PRESETS = {
    "es": {"name": "ESPAÑOL",   "default_voice": "es-ES-AlvaroNeural", "onscreen_case": "UPPER"},
    "en": {"name": "ENGLISH",   "default_voice": "en-US-GuyNeural",    "onscreen_case": "UPPER"},
    "fr": {"name": "FRANÇAIS",  "default_voice": "fr-FR-HenriNeural",  "onscreen_case": "UPPER"},
    "pt": {"name": "PORTUGUÊS", "default_voice": "pt-BR-AntonioNeural","onscreen_case": "UPPER"},
    "ko": {"name": "KOREAN",    "default_voice": "ko-KR-InJoonNeural", "onscreen_case": "NORMAL"},
    "ar": {"name": "ARABIC",    "default_voice": "ar-EG-ShakirNeural", "onscreen_case": "NORMAL"},
}

# Variables globales (las leerán tus prompts)
VIDEO_LANG_CODE = "es"
VIDEO_LANG_NAME = "ESPAÑOL"
VIDEO_ONSCREEN_CASE = "UPPER"  # UPPER o NORMAL

def detect_video_language_from_title(title: str) -> str:
    t = (title or "").strip()
    if not t:
        return "es"
    try:
        code = detect(t)
    except Exception:
        code = "es"

    # Normaliza a tu set soportado
    if code.startswith("es"): return "es"
    if code.startswith("en"): return "en"
    if code.startswith("fr"): return "fr"
    if code.startswith("pt"): return "pt"
    if code.startswith("ko"): return "ko"
    if code.startswith("ar"): return "ar"

    # fallback si detecta algo raro
    return "es"

def set_video_language_from_title(title: str) -> str:
    global VIDEO_LANG_CODE, VIDEO_LANG_NAME, VIDEO_ONSCREEN_CASE
    code = detect_video_language_from_title(title)
    preset = LANG_PRESETS.get(code, LANG_PRESETS["es"])
    VIDEO_LANG_CODE = code
    VIDEO_LANG_NAME = preset["name"]
    VIDEO_ONSCREEN_CASE = preset.get("onscreen_case", "UPPER")
    return code

def resolve_voice_id(voice_label: str, title: str) -> str:
    """
    Si el usuario eligió AUTO, selecciona voz por idioma del título.
    Si eligió voz concreta, usa esa.
    """
    v = VOICE_OPTIONS.get(voice_label, "AUTO")
    if v != "AUTO":
        return v

    code = set_video_language_from_title(title)
    return LANG_PRESETS[code]["default_voice"]



_log_cell2("\nCELDA 2 COMPLETADA.")
_log_cell2(f"sd-cli: {SDCLI_PATH} (existe={os.path.exists(SDCLI_PATH)}) | backend compilado: {SDCLI_BACKEND} | gpu_runtime: {SDCLI_GPU_AVAILABLE}")
_log_cell2(f"CPU_THREADS={CPU_THREADS} | SDCLI_THREADS={SDCLI_THREADS}")
_log_cell2(f"Backends de imagen: {list(IMAGE_BACKENDS.keys())}")
_log_cell2(f"Presets Qwen local disponibles ({len(QWEN21_LOCAL_PRESETS)}).")
_log_cell2(f"Preset por defecto: {DEFAULT_QWEN21_PRESET_LABEL}")

[16:49:21] Plataforma detectada: kaggle
[16:49:21] OUTPUT_DIR: /kaggle/working/VideoAutomationProject
[16:49:21] SDCPP_DIR: /kaggle/working/stable-diffusion.cpp
[16:49:21] LOG_FILE: /kaggle/working/VideoAutomationProject/run.log


<IPython.core.display.Javascript object>

[16:49:21] Anti-desconexión (Colab) inyectado.
[16:49:21] 
CELDA 2 COMPLETADA.
[16:49:21] sd-cli: /kaggle/working/stable-diffusion.cpp/build/bin/sd-cli (existe=True) | backend compilado: cuda | gpu_runtime: True
[16:49:21] CPU_THREADS=2 | SDCLI_THREADS=4
[16:49:21] Backends de imagen: ['Pollinations (API gratis, rápido)', 'Qwen-Image-2.1 Local (GGUF, sd-cli) [GPU rápido / CPU lento, según preset]']
[16:49:21] Presets Qwen local disponibles (14).
[16:49:21] Preset por defecto: Diffusion Q8_0 + Encoder Q4_K_M (~13.4GB total, encoder oficial/Qwen) [DIFUSIÓN NO CONFIRMADA EN LISTADO — puede fallar la descarga]
[16:49:38]    429 detectado en mistralai/mistral-large-2512. Esperando 10.0s...
[16:49:50]    429 detectado en mistralai/mistral-large-2512. Esperando 20.0s...


In [13]:
# ═══════════════════════════════════════════════════════════════
# CELDA UTILIDAD: LIMPIEZA COMPLETA DEL RUN (sin tocar sd-cli ni modelos)
# - Borra persistencia de estado (checkpoint/project_state)
# - Borra imágenes, audios, clips por escena, video final, previews, prompts txt, logs (opcional)
# - NO borra: /kaggle/working/stable-diffusion.cpp (compilación)
# - NO borra: OUTPUT_DIR/local_models (GGUF/VAE cache persistente)
# ═══════════════════════════════════════════════════════════════

import os, shutil, glob

def _rm(path):
    if not path:
        return
    try:
        if os.path.isdir(path):
            shutil.rmtree(path, ignore_errors=True)
            print(f"DIR borrada: {path}")
        elif os.path.exists(path):
            os.remove(path)
            print(f"Archivo borrado: {path}")
    except Exception as e:
        print(f"No se pudo borrar {path}: {e}")

def _rm_glob(pattern):
    for p in glob.glob(pattern):
        _rm(p)

print("OUTPUT_DIR:", OUTPUT_DIR)

# 1) Estado/persistencia del proyecto (para que regenere prompts/escenas desde cero)
for p in [CHECKPOINT_PATH, PROJECT_STATE_PATH]:
    _rm(p)

# 2) Carpetas grandes generadas por el flujo
#    (NO borres local_models porque ahí están los GGUF/vae en caché)
folders_to_delete = [
    os.path.join(OUTPUT_DIR, "images"),
    os.path.join(OUTPUT_DIR, "audio_chunks"),
    os.path.join(OUTPUT_DIR, "scene_audio_clips"),
]

for d in folders_to_delete:
    _rm(d)
    os.makedirs(d, exist_ok=True)  # las recreamos vacías para evitar errores posteriores
    print(f"DIR recreada vacía: {d}")

# 3) Archivos grandes típicos en raíz del proyecto
files_to_delete = [
    os.path.join(OUTPUT_DIR, "full_narration.wav"),
    os.path.join(OUTPUT_DIR, "full_narration_original.wav"),
    os.path.join(OUTPUT_DIR, "_full_narration_raw.wav"),
    os.path.join(OUTPUT_DIR, "_full_narration_from_clips.wav"),
    os.path.join(OUTPUT_DIR, "final_video.mp4"),
    os.path.join(OUTPUT_DIR, "script.txt"),
]

for f in files_to_delete:
    _rm(f)

# 4) Prompts exportados
_rm_glob(os.path.join(OUTPUT_DIR, "image_prompts_*.txt"))

# 5) Logs (opcional). Si quieres conservar run.log, comenta esta línea.
_rm(os.path.join(OUTPUT_DIR, "run.log"))

# 6) Limpieza del "Editor Pro" integrado (si lo usaste)
_rm(os.path.join(OUTPUT_DIR, "pro_editor"))

print("\n✅ Limpieza completada.")
print("✅ NO se tocó stable-diffusion.cpp (sd-cli compilado).")
print("✅ NO se tocó local_models (modelos GGUF/VAE).")
print("Ahora ejecuta PASO 1 otra vez para generar un proyecto nuevo desde cero.")

OUTPUT_DIR: /kaggle/working/VideoAutomationProject
Archivo borrado: /kaggle/working/VideoAutomationProject/checkpoint.json
Archivo borrado: /kaggle/working/VideoAutomationProject/project_state.json
DIR borrada: /kaggle/working/VideoAutomationProject/images
DIR recreada vacía: /kaggle/working/VideoAutomationProject/images
DIR borrada: /kaggle/working/VideoAutomationProject/audio_chunks
DIR recreada vacía: /kaggle/working/VideoAutomationProject/audio_chunks
DIR borrada: /kaggle/working/VideoAutomationProject/scene_audio_clips
DIR recreada vacía: /kaggle/working/VideoAutomationProject/scene_audio_clips
Archivo borrado: /kaggle/working/VideoAutomationProject/full_narration.wav
Archivo borrado: /kaggle/working/VideoAutomationProject/full_narration_original.wav
Archivo borrado: /kaggle/working/VideoAutomationProject/script.txt
Archivo borrado: /kaggle/working/VideoAutomationProject/run.log
DIR borrada: /kaggle/working/VideoAutomationProject/pro_editor

✅ Limpieza completada.
✅ NO se tocó sta

In [15]:
# ═══════════════════════════════════════════════════════════════
# CELDA 3: MOTOR (xKiro + Pollinations + Qwen local) - MEJORADA
# TODAS las funcionalidades originales + timing exacto + audio supervisable
# ═══════════════════════════════════════════════════════════════

import os, re, gc, json, subprocess, asyncio, traceback, time, random, tempfile, math
import urllib.parse, requests
from dataclasses import dataclass, asdict
from typing import List, Dict, Optional, Tuple

import torch
import edge_tts
from openai import OpenAI

try:
    import psutil
except ImportError:
    psutil = None

# ---------------------------------------------------------------
# CONFIG TIMING
# ---------------------------------------------------------------
FPS = 30
LEAD_MS = 120
LEAD_S = LEAD_MS / 1000.0
MIN_SCENE_S = 0.15

SILENCE_NOISE_DB = "-35dB"
SILENCE_MIN_DUR_S = 0.15
SILENCE_MATCH_TOL_S = 0.80

PROJECT_LOG_FILE = os.environ.get("PROJECT_LOG_FILE", f"{OUTPUT_DIR}/run.log")

def _append_log(line: str):
    try:
        with open(PROJECT_LOG_FILE, "a", encoding="utf-8") as f:
            f.write(line.rstrip() + "\n")
    except Exception:
        pass

def _log(msg: str) -> None:
    ts = time.strftime("%H:%M:%S")
    line = f"[{ts}] {msg}"
    print(line)
    _append_log(line)

# ---------------------------------------------------------------
# Utilidades de timing exacto
# ---------------------------------------------------------------
def _ffprobe_duration(path: str) -> Optional[float]:
    try:
        cmd = ["ffprobe", "-v", "error", "-show_entries", "format=duration",
               "-of", "default=noprint_wrappers=1:nokey=1", path]
        r = subprocess.run(cmd, capture_output=True, text=True)
        return float((r.stdout or "").strip())
    except Exception:
        return None

def normalize_audio_to_frame_grid(audio_in: str, audio_out: str, fps: int = FPS) -> Tuple[float, float]:
    d = _ffprobe_duration(audio_in) or 0.0
    if d <= 0:
        shutil.copyfile(audio_in, audio_out)
        return 0.0, 0.0
    target = round(d * fps) / fps
    cmd = (
        f"ffmpeg -y -i '{audio_in}' "
        f"-af 'apad=pad_dur=10,atrim=0:{target:.6f}' "
        f"-acodec pcm_s16le -ar 24000 '{audio_out}'"
    )
    subprocess.run(cmd, shell=True, capture_output=True, text=True)
    d2 = _ffprobe_duration(audio_out) or target
    return d, d2

def detect_silence_end_times(audio_path: str,
                             noise_db: str = SILENCE_NOISE_DB,
                             min_dur_s: float = SILENCE_MIN_DUR_S) -> List[float]:
    cmd = f"ffmpeg -hide_banner -i '{audio_path}' -af silencedetect=noise={noise_db}:d={min_dur_s} -f null -"
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    text = (r.stderr or "") + "\n" + (r.stdout or "")
    ends = []
    for m in re.finditer(r"silence_end:\s*([0-9]*\.?[0-9]+)", text):
        try:
            ends.append(float(m.group(1)))
        except Exception:
            pass
    ends = sorted(set(ends))
    _log(f"[silencedetect] silence_end detectados: {len(ends)} (noise={noise_db}, d={min_dur_s})")
    return ends

def match_silence_end(target_t: float, silence_ends: List[float], tol_s: float = SILENCE_MATCH_TOL_S) -> Optional[float]:
    if not silence_ends:
        return None
    best = None
    best_dt = 1e9
    for t in silence_ends:
        dt = abs(t - target_t)
        if dt <= tol_s and dt < best_dt:
            best = t
            best_dt = dt
    return best

def compute_cutpoints_from_segments(segments: List[Dict],
                                   audio_duration: float,
                                   lead_s: float = LEAD_S) -> List[float]:
    n = len(segments)
    cps = [0.0]
    for i in range(1, n):
        target = float(segments[i]["start"])
        cut = max(target - lead_s, cps[-1] + MIN_SCENE_S)
        cps.append(cut)
    cps.append(float(audio_duration))
    return cps

def refine_cutpoints_with_silences(cutpoints: List[float],
                                  segments: List[Dict],
                                  silence_ends: List[float],
                                  lead_s: float = LEAD_S) -> List[float]:
    if not silence_ends:
        return cutpoints

    refined = [cutpoints[0]]
    for i in range(1, len(cutpoints) - 1):
        target_start = float(segments[i]["start"])
        se = match_silence_end(target_start, silence_ends, tol_s=SILENCE_MATCH_TOL_S)
        if se is None:
            new_cut = max(target_start - lead_s, refined[-1] + MIN_SCENE_S)
        else:
            new_cut = max(se - lead_s, refined[-1] + MIN_SCENE_S)
        refined.append(new_cut)
    refined.append(cutpoints[-1])
    return refined

def cutpoints_to_frame_durations(cutpoints: List[float], fps: int = FPS) -> List[float]:
    total = cutpoints[-1] - cutpoints[0]
    total_frames_target = round(total * fps)

    accumulated_ideal = 0.0
    accumulated_real = 0
    durs = []

    for i in range(len(cutpoints) - 1):
        dur = cutpoints[i+1] - cutpoints[i]
        accumulated_ideal += dur * fps
        target_frames_so_far = round(accumulated_ideal)
        frames_this = max(1, target_frames_so_far - accumulated_real)
        accumulated_real += frames_this
        durs.append(frames_this / fps)

    diff = total_frames_target - accumulated_real
    if diff != 0:
        durs[-1] += diff / fps

    return durs

def _build_atempo_chain(ratio: float) -> str:
    if ratio <= 0:
        return "atempo=1.0"
    parts = []
    r = ratio
    while r < 0.5:
        parts.append("atempo=0.5")
        r /= 0.5
    while r > 2.0:
        parts.append("atempo=2.0")
        r /= 2.0
    parts.append(f"atempo={r:.6f}")
    return ",".join(parts)

# ---------------------------------------------------------------
# Scene dataclass (mejorado, campos originales + nuevos)
# ---------------------------------------------------------------
@dataclass
class Scene:
    scene_id: int
    narration_text: str
    image_prompt: str = ""

    audio_start: float = 0.0
    audio_end: float = 0.0

    visual_start: float = 0.0
    visual_end: float = 0.0

    duration: float = 0.0

    audio_clip_path: Optional[str] = None
    image_path: Optional[str] = None

    status: str = "pending"
    audio_status: str = "pending"

# ---------------------------------------------------------------
# Helpers originales (sin cambios funcionales)
# ---------------------------------------------------------------
def _slugify(text: str) -> str:
    s = re.sub(r'[^a-zA-Z0-9\s-]', '', text or "").strip().lower()
    s = re.sub(r'[\s-]+', '_', s)
    return s[:60] if s else "video"

def _word_count(text: str) -> int:
    return len((text or "").split())

def _extract_retry_seconds(err_str: str) -> Optional[float]:
    m = re.search(r"retry in\s+([\d.]+)s", err_str, re.I)
    if m:
        try: return float(m.group(1))
        except: return None
    return None

def build_script_system_prompt(target_words: int, duration_min: float) -> str:
    return f"""Eres un guionista experto en videos virales de YouTube sobre historia humana, evolución, antropología y psicología.

REGLA DE IDIOMA (OBLIGATORIA):
- Escribe TODO el guion en: {VIDEO_LANG_NAME}.
- NO mezcles idiomas.
- Usa una redacción natural y nativa en {VIDEO_LANG_NAME}.

REGLAS OBLIGATORIAS:
1. Longitud: Aproximadamente {target_words} palabras (para {duration_min} minutos a ~150 palabras/minuto).
2. Voz: Usa SIEMPRE la segunda persona ("tú", "tu cerebro", "tus antepasados"). JAMÁS uses "nosotros" ni "yo".
3. Gancho: Abre con un momento moderno cotidiano que haga reflexionar sobre algo que nadie conoce → inmediatamente di "pero la respuesta real es mucho más extraña" → replantea todo por completo.
4. Ritmo: Oración corta. Oración corta. Una oración más larga que construye profundidad. Oración corta. Una pregunta pertinente cada 4-6 oraciones.
5. Arco narrativo: Gancho → Replanteamiento → Inmersión histórica/científica profunda → Giro contraintuitivo → Espejo moderno → Línea de cierre que resuena con la apertura inicial.
6. Lenguaje: Español sencillo. Sin jerga sin explicar. Cada término científico se descodifica inmediatamente.
7. Contenido: Incluye al menos 3 investigadores o estudios reales nombrados e integrados de forma natural y fluida.
8. Cierre: Termina reflejando la verdad antigua o científica sobre algo que el espectador siente o hace hoy en su vida cotidiana moderna.
9. Formato: SOLO texto plano continuo. Sin markdown, sin asteriscos, sin títulos, sin indicaciones visuales, sin notas entre paréntesis, sin acotaciones de dirección.
"""

def build_image_style_guide_block() -> str:
    framing_lines = "\n".join(f"- {t}" for t in STYLE_GUIDE["framing_types"])
    return (
        "ESTILO VISUAL DEL CANAL (sigue esto estrictamente en TODAS las escenas):\n"
        f"- Arte: {STYLE_GUIDE['art_style']}\n"
        f"- Personajes: {STYLE_GUIDE['characters']}\n"
        f"- Objetos/Animales: {STYLE_GUIDE['objects']}\n"
        f"- Fondos: {STYLE_GUIDE['backgrounds']}\n"
        f"- Texto en pantalla: {STYLE_GUIDE['text_rules']}\n"
        f"- Etiquetas/Flechas: {STYLE_GUIDE['labels_arrows']}\n"
        f"- Bocadillos de pensamiento: {STYLE_GUIDE['thought_bubbles']}\n"
        f"- Paleta de colores permitida: {STYLE_GUIDE['color_palette']}\n"
        f"- Relación de aspecto: {STYLE_GUIDE['aspect_ratio']}\n"
        f"- Elementos a evitar SIEMPRE: {STYLE_GUIDE['negative_prompt']}\n\n"
        "TIPOS DE ENCUADRE DISPONIBLES (elige el más apropiado según la narración):\n"
        f"{framing_lines}\n\n"
        "REGLA DE TONO -> COLOR DE FONDO:\n"
        "- Antiguo/prehistórico -> fondo beige o azul oscuro,monigotes siempre vestidos con pieles de color marron y cabello\n"
        "- Peligro/amenaza -> fondo de color claro pero nunca blanco con texto rojo, o cielo teñido de rojo\n"
        "- Feliz/triunfo/descubrimiento -> fondo de tono claro  o amarillo, nunca blanco\n"
        "- Bajo el agua/ciencia -> fondo azul sólido\n"
        "- Exterior/naturaleza/evolución -> suelo verde con cespet + cielo azul con nuves, fondo con Arboles y paisajes detallado\n"
        "- Fuego/noche/ritual antiguo -> fondo naranja sólido\n"
        "- Escena cotidiana/moderna/actualidad -> fondo que ilustre un abiente moderno ciudad,casas,abitaciones, interiores de hogares,exteriores hurbanos, elementos de la vida cotidiana como electrodomesticos o hutencilios y herramientas actuales y cotidianas\n"
        "- Al mencionar a un cientifico reconosido -> ambiente moderno, de universitario, lavoratorio con elementos y fondo que den contexto a la escena de la forma que mejor corresponda\n"
        "- al mencionar allasgos arqueologicos o descumbrimientos sientificos -> escena que ilustre el suseso con de la forma mas combinsente y que mejor pueda mostrar el allasgo\n"
        "- al ablar de animales -> ilustrar el animal con caricaturas bastante realistas\n"
        "- al apareser un monigote y fondo claro -> el monigote tiene que que estar completamente definido con lideas negras tanto su silueta y contorno como los demas elementos de la escena, El contraste y color del fondo nunca ser blanco brillate "
        
    )

def build_image_system_prompt() -> str:
     # Valores seguros por defecto si todavía no se detectó el idioma del título
    video_lang_name = globals().get("VIDEO_LANG_NAME", "ESPAÑOL")
    video_onscreen_case = globals().get("VIDEO_ONSCREEN_CASE", "UPPER")

    # Esta variable debe estar definida DENTRO de la función y antes del return
    casing_rule = (
        "El texto visible en pantalla debe escribirse en MAYÚSCULAS."
        if video_onscreen_case == "UPPER"
        else
        "El texto visible en pantalla debe usar escritura normal. "
        "NO fuerces las mayúsculas para este idioma."
    )
    
    return (
        "Eres el director de arte de un canal de YouTube de animación  2D "
        "educativos sobre historia humana, evolución, antropología y psicología.\n\n"
        "Tu única tarea es escribir UN prompt de generación de imagen en INGLÉS para la "
        "escena que se te solicite, siguiendo estrictamente el ADN visual del canal y el "
        "contexto narrativo completo que se te proporcione.\n\n"
        f"{build_image_style_guide_block()}\n\n"
        "REGLAS OBLIGATORIAS DEL PROMPT:\n"
        "1. El prompt debe empezar EXACTAMENTE con: \"Detailed 2D cartoon animation style, simple flat textures, thick and well-defined black lines with hand-drawn marker strokes, featuring rich and detailed background elements that naturally correspond to and enhance the main scene.\"\n"
        "2. El prompt debe terminar EXACTAMENTE con: \" 16:9 aspect ratio, educational YouTube style.\"\n"
        "3. Entre esas dos frases, sé MUY específico: qué personajes hay, qué hacen, su expresión, que elementos tienen que apareser en la ilustración y el lugar exacto donde tienen que situarse "
        "exacta, qué objetos aparecen, el color de fondo elegido según la regla de tono y los elementos existentes en el fondo que ilustren el contexto de la escena como un paisaje cuevas etc, pero si se abla de un escenario moderno o cotidiano los elementos y el fondo tiene que describir escenas modernas y actuales con elementos que roda persona conose en su día a día, y "
        "cualquier texto/etiqueta en pantalla.\n"
        "4. Traduce narración abstracta en imágenes concretas usando los monigotes (stick figures).\n"
        "5. Si esta escena continúa el mismo momento que la escena anterior (mismo lugar/acción), "
        "MANTÉN los mismos elementos base y cambia solo la expresión o añade un elemento pequeño. "
        "NO inventes una escena completamente distinta sin motivo.\n"
        f"6. Todo el prompt debe estar en INGLÉS. Si la escena requiere texto visible en la imagen "
        f"(letrero, etiqueta, bocadillo de pensamiento), ese texto debe escribirse en {VIDEO_LANG_NAME},"
        f"e indicarse explícitamente entre comillas dentro de la descripción. {casing_rule}\n" 
        "7. NUNCA menciones marcas de tiempo.\n"
        "8. Mínimo 120 palabras, Máximo 160 palabras. Devuelve SOLO el prompt final en una sola línea."
    )

def build_image_prompt_request(full_script: str, segment_text: str, duration: float,
                                scene_index: int, total_scenes: int,
                                previous_prompts: List[str]) -> str:
    context_block = (full_script or "").strip()
    if len(context_block) > 6000:
        context_block = context_block[:6000] + " ..."

    lines = []
    lines.append("GUION COMPLETO DEL VIDEO:")
    lines.append("---INICIO GUION---")
    lines.append(context_block if context_block else "(guion no disponible)")
    lines.append("---FIN GUION---")

    if previous_prompts:
        lines.append("")
        lines.append("CONTINUIDAD VISUAL - prompts previos:")
        for p in previous_prompts:
            lines.append(f"- {p}")

    lines.append("")
    lines.append(f"ESCENA {scene_index}/{total_scenes} | duración {duration:.2f}s")
    lines.append("TEXTO EXACTO DEL NARRADOR:")
    lines.append(f'"{segment_text}"')
    lines.append("")
    lines.append("Genera ahora el prompt de imagen para ESTA escena.")
    return "\n".join(lines)

def sanitize_image_prompt(raw: str) -> str:
    text = (raw or "").strip().strip('"').strip("'")
    text = re.sub(r'\[\d{1,2}:\d{2}(:\d{2})?\]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()

    anchor = (
        "Detailed 2D cartoon animation style, simple flat textures, thick and well-defined black lines "
        "with hand-drawn marker strokes, featuring rich and detailed background elements that naturally "
        "correspond to and enhance the main scene."
    )
    closer = "16:9 aspect ratio, educational YouTube style."
      # Si el modelo ya incluyó el anchor en medio, recorta para que empiece ahí
    if anchor in text and not text.startswith(anchor):
        text = text[text.find(anchor):].strip()

    # Forzar inicio EXACTO
    if not text.startswith(anchor):
        text = f"{anchor} {text}".strip()

    # Si el closer aparece antes, recorta para que termine exactamente ahí (evita duplicados)
    if closer in text:
        text = text[: text.rfind(closer) + len(closer)].strip()

    # Forzar final EXACTO
    if not text.endswith(closer):
        text = f"{text.rstrip()} {closer}".strip()

    return text

# ---------------------------------------------------------------
# Motor principal
# ---------------------------------------------------------------
class VideoAutomationEngine:
    SDCLI_OPTIONAL_FLAGS = ["--diffusion-fa", "--mmap", "--vae-tiling",
                            "--threads", "--offload-to-cpu", "--params-backend"]

    def __init__(self, xkiro_api_key: str, xkiro_base_url: str = XKIRO_BASE_URL,
                 output_dir: str = OUTPUT_DIR):
        self.client = OpenAI(api_key=xkiro_api_key, base_url=xkiro_base_url)
        self.output_dir = output_dir

        self.scenes: List[Scene] = []
        self.script_text: str = ""
        self.current_theme: str = ""
        self.theme_slug: str = "video"

        self.full_audio_path: str = f"{output_dir}/full_narration.wav"
        self.full_audio_original_path: str = f"{output_dir}/full_narration_original.wav"
        self.scene_audio_dir: str = f"{output_dir}/scene_audio_clips"
        os.makedirs(self.scene_audio_dir, exist_ok=True)

        self.prompts_preview_path: Optional[str] = None

        self.text_model: str = DEFAULT_XKIRO_TEXT_MODEL_ID
        self.text_fallback_pool: List[str] = XKIRO_FALLBACK_POOL

        self.image_backend: str = "pollinations"
        self.pollinations_model: str = "flux"

        self.qwen_local_cfg: Optional[dict] = None
        self.qwen_local_cfg_label: Optional[str] = None

        self.sdcli_flags: Optional[set] = None

        self._try_load_checkpoint()

    # ------------------ Persistencia de proyecto ------------------
    def _try_load_checkpoint(self):
        try:
            if os.path.exists(CHECKPOINT_PATH):
                with open(CHECKPOINT_PATH, "r", encoding="utf-8") as f:
                    data = json.load(f)
                scenes = []
                for s in data:
                    scenes.append(Scene(**s))
                self.scenes = scenes
                _log(f"[Persistencia] checkpoint.json cargado: {len(self.scenes)} escenas")
        except Exception as e:
            _log(f"[Persistencia] No se pudo cargar checkpoint: {e}")

        try:
            if os.path.exists(PROJECT_STATE_PATH):
                with open(PROJECT_STATE_PATH, "r", encoding="utf-8") as f:
                    st = json.load(f)
                self.script_text = st.get("script_text", "") or self.script_text
                self.current_theme = st.get("current_theme", "") or self.current_theme
                self.theme_slug = st.get("theme_slug", "video") or self.theme_slug
                _log(f"[Persistencia] project_state.json cargado (theme_slug={self.theme_slug})")
        except Exception as e:
            _log(f"[Persistencia] No se pudo cargar project_state: {e}")

    def _save_state(self):
        try:
            with open(CHECKPOINT_PATH, "w", encoding="utf-8") as f:
                json.dump([asdict(s) for s in self.scenes], f, ensure_ascii=False, indent=2)
        except Exception as e:
            _log(f"[Persistencia] Error guardando checkpoint: {e}")

        try:
            with open(PROJECT_STATE_PATH, "w", encoding="utf-8") as f:
                json.dump({
                    "script_text": self.script_text,
                    "current_theme": self.current_theme,
                    "theme_slug": self.theme_slug,
                    "full_audio_path": self.full_audio_path,
                    "updated_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
                }, f, ensure_ascii=False, indent=2)
        except Exception as e:
            _log(f"[Persistencia] Error guardando project_state: {e}")

    # ------------------ sd-cli help/flags ------------------
    def _sdcli_help_text(self) -> str:
        candidates = [
            [SDCLI_PATH, "--help"],
            [SDCLI_PATH, "-?"],
            [SDCLI_PATH],
        ]
        for args in candidates:
            try:
                r = subprocess.run(args, capture_output=True, text=True, timeout=30)
                txt = (r.stdout or "") + "\n" + (r.stderr or "")
                if "Usage:" in txt or "CLI Options" in txt or "--diffusion-model" in txt:
                    return txt
            except Exception:
                pass
        try:
            r = subprocess.run([SDCLI_PATH, "-h"], capture_output=True, text=True, timeout=10)
            return (r.stdout or "") + "\n" + (r.stderr or "")
        except Exception:
            return ""

    def _detect_sdcli_flags(self) -> set:
        if self.sdcli_flags is not None:
            return self.sdcli_flags
        flags = set()
        try:
            txt = self._sdcli_help_text()
            if txt:
                flags = set(re.findall(r'(--[a-zA-Z0-9][a-zA-Z0-9_-]*)', txt))
        except Exception as e:
            _log(f"   No se pudo leer help de sd-cli: {e}")
        self.sdcli_flags = flags
        if flags:
            _log(f"   sd-cli help: {len(flags)} flags detectados")
            for fl in self.SDCLI_OPTIONAL_FLAGS:
                _log(f"     {fl}: {'SÍ' if fl in flags else 'NO (se omitirá)'}")
        else:
            _log("   Aviso: no se detectaron flags de sd-cli; se enviarán sin filtrar.")
        return flags

    def _sdcli_supports(self, flag: str) -> bool:
        flags = self._detect_sdcli_flags()
        return (not flags) or (flag in flags)

    # ------------------ Model switching ------------------
    def set_text_model(self, model_id: str):
        if model_id != self.text_model:
            _log(f"Cambiando modelo de texto xKiro: {self.text_model} → {model_id}")
            self.text_model = model_id

    def set_image_backend(self, backend_key: str):
        if backend_key != self.image_backend:
            _log(f"Cambiando backend de imagen: {self.image_backend} → {backend_key}")
            self.image_backend = backend_key

    def set_image_model(self, model_key: str):
        if model_key != self.pollinations_model:
            _log(f"Cambiando modelo Pollinations: {self.pollinations_model} → {model_key}")
            self.pollinations_model = model_key

    def set_qwen_local_preset(self, preset_label: str):
        if self.qwen_local_cfg is not None and self.qwen_local_cfg_label == preset_label:
            _log(f"Preset Qwen local ya preparado: {preset_label} (reutilizando)")
            return
        _log(f"Preparando preset Qwen local: {preset_label}")
        self.qwen_local_cfg = ensure_qwen21_local_files(preset_label)
        self.qwen_local_cfg_label = preset_label
        _log(f"Qwen local listo. sd-cli backend: {SDCLI_BACKEND} | nvidia-smi visible: {SDCLI_GPU_AVAILABLE}")
        self._detect_sdcli_flags()

    # ------------------ xKiro fallback ------------------
    def _call_xkiro_with_fallback(self, system_instruction: str,
                                   user_content: str,
                                   temperature: float,
                                   max_output_tokens: int) -> str:
        candidate_models = [self.text_model] + [m for m in self.text_fallback_pool if m != self.text_model]
        messages = [
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": user_content},
        ]
        last_err = None

        for model in candidate_models:
            for attempt in range(1, 4):
                try:
                    resp = self.client.chat.completions.create(
                        model=model,
                        messages=messages,
                        temperature=float(temperature),
                        max_tokens=int(max_output_tokens),
                    )
                    text = (resp.choices[0].message.content or "").strip()
                    if not text:
                        raise RuntimeError("Respuesta vacía del modelo")
                    return text
                except Exception as e:
                    last_err = e
                    err_str = str(e)
                    if "429" in err_str:
                        wait_s = _extract_retry_seconds(err_str) or (10.0 * attempt)
                        _log(f"   429 detectado en {model}. Esperando {wait_s:.1f}s...")
                        time.sleep(wait_s)
                        continue
                    if "503" in err_str or "UNAVAILABLE" in err_str:
                        backoff = 5.0 * attempt
                        _log(f"   503/UNAVAILABLE en {model}. Esperando {backoff:.1f}s...")
                        time.sleep(backoff)
                        continue
                    _log(f"   fallo en {model}: {e}")
                    break

        raise RuntimeError(f"Todos los modelos xKiro fallaron. Último error: {last_err}")

    # ------------------ Script ------------------
    def generate_script(self, theme: str, duration_min: float) -> str:
        t0 = time.time()
        self.current_theme = theme
        self.theme_slug = _slugify(theme)
        _log(f"[FASE 2] Generando guion: '{theme}' ({duration_min} min)")
        target_words = int(duration_min * 150)
        _log(f"   Objetivo: ~{target_words} palabras | modelo principal: {self.text_model}")
        system_prompt = build_script_system_prompt(target_words, duration_min)

        script = self._call_xkiro_with_fallback(
            system_instruction=system_prompt,
            user_content=f"Tema del video: {theme}",
            temperature=0.7,
            max_output_tokens=4000,
        )
        script = re.sub(r'[*#_`\[\]]', '', script)
        self.script_text = script

        with open(f"{self.output_dir}/script.txt", "w", encoding="utf-8") as f:
            f.write(script)

        elapsed = time.time() - t0
        _log(f"   Guion generado: {_word_count(script)} palabras en {elapsed:.1f}s")
        _log(f"   Preview: \"{script[:120].strip()}...\"")
        self._save_state()
        return script

    # ------------------ Audio (TTS) ------------------
    def _split_into_chunks(self, text: str, max_chars: int = 500) -> List[str]:
        sentences = re.split(r'(?<=[.!?])\s+', text)
        chunks, current = [], ""
        for sent in sentences:
            if len(current) + len(sent) + 1 <= max_chars:
                current = f"{current} {sent}".strip()
            else:
                if current:
                    chunks.append(current)
                current = sent
        if current:
            chunks.append(current)
        return chunks

    def _get_media_duration(self, path: str) -> Optional[float]:
        return _ffprobe_duration(path)

    def generate_audio(self, script: str, voice: str = "es-ES-AlvaroNeural") -> str:
        t0 = time.time()
        _log(f"[FASE 3] Generando audio con voz: {voice}")
        chunks = self._split_into_chunks(script, max_chars=500)
        _log(f"   Guion dividido en {len(chunks)} fragmentos")

        chunk_dir = f"{self.output_dir}/audio_chunks"
        os.makedirs(chunk_dir, exist_ok=True)
        chunk_paths = []

        async def gen_chunk(text: str, out_path: str, voice: str):
            communicate = edge_tts.Communicate(text, voice)
            await communicate.save(out_path)

        for i, chunk in enumerate(chunks):
            out_path = f"{chunk_dir}/chunk_{i:04d}.wav"
            asyncio.run(gen_chunk(chunk, out_path, voice))
            chunk_paths.append(out_path)
            _log(f"   Fragmento {i+1}/{len(chunks)} generado ({len(chunk)} caracteres)")

        concat_list_path = f"{chunk_dir}/concat_list.txt"
        with open(concat_list_path, "w", encoding="utf-8") as f:
            for p in chunk_paths:
                f.write(f"file '{os.path.abspath(p)}'\n")

        tmp_full = f"{self.output_dir}/_full_narration_raw.wav"
        cmd = f"ffmpeg -y -f concat -safe 0 -i '{concat_list_path}' -acodec pcm_s16le -ar 24000 '{tmp_full}'"
        subprocess.run(cmd, shell=True, capture_output=True, text=True)

        try:
            shutil.copyfile(tmp_full, self.full_audio_original_path)
        except Exception:
            pass

        d0, d1 = normalize_audio_to_frame_grid(tmp_full, self.full_audio_path, fps=FPS)
        try:
            os.remove(tmp_full)
        except Exception:
            pass

        for p in chunk_paths:
            try: os.remove(p)
            except Exception: pass

        elapsed = time.time() - t0
        _log(f"   Audio completo: {self.full_audio_path} | dur_raw={d0:.3f}s | dur_norm={d1:.3f}s | tiempo proceso: {elapsed:.1f}s")
        self._save_state()
        return self.full_audio_path

    # ------------------ WhisperX ------------------
    def transcribe_and_align(self, audio_path: str) -> List[Dict]:
        t0 = time.time()
        _log(f"[FASE 4] Transcribiendo con WhisperX large-v3...")
        import whisperx

        device = "cuda" if torch.cuda.is_available() else "cpu"
        compute_type = "float16" if device == "cuda" else "int8"
        _log(f"   Device (whisperx): {device} | compute_type: {compute_type}")

        whisper_model = None
        align_model = None
        try:
            whisper_model = whisperx.load_model("large-v3", device=device, compute_type=compute_type, vad_method="silero")
            audio = whisperx.load_audio(audio_path)
            result = whisper_model.transcribe(audio, batch_size=16)
            _log(f"   Transcripción cruda: {len(result['segments'])} segmentos detectados")

            align_model, metadata = whisperx.load_align_model(language_code="es", device=device)
            result = whisperx.align(result["segments"], align_model, metadata, audio, device=device)

            grouped = []
            for seg in result["segments"]:
                grouped.append({"text": seg["text"].strip(), "start": float(seg["start"]), "end": float(seg["end"])})

            elapsed = time.time() - t0
            total_dur = grouped[-1]["end"] - grouped[0]["start"] if grouped else 0
            avg_dur = (total_dur / len(grouped)) if grouped else 0
            _log(f"   Alineación completada: {len(grouped)} segmentos finales | duración media/segmento: {avg_dur:.2f}s | tiempo: {elapsed:.1f}s")
            return grouped

        finally:
            try: del whisper_model
            except Exception: pass
            try: del align_model
            except Exception: pass
            torch.cuda.empty_cache()
            gc.collect()

    # ------------------ Construcción de escenas con timing exacto ------------------
    def build_scenes_with_exact_timing(self, segments: List[Dict]) -> None:
        if not segments:
            raise RuntimeError("No hay segmentos para construir escenas")

        audio_dur = self._get_media_duration(self.full_audio_path) or float(segments[-1]["end"])
        _log(f"[TIMING] Duración audio (ffprobe): {audio_dur:.6f}s | FPS={FPS} | LEAD={LEAD_MS}ms")

        cutpoints = compute_cutpoints_from_segments(segments, audio_dur, lead_s=LEAD_S)
        silence_ends = detect_silence_end_times(self.full_audio_path, noise_db=SILENCE_NOISE_DB, min_dur_s=SILENCE_MIN_DUR_S)
        cutpoints = refine_cutpoints_with_silences(cutpoints, segments, silence_ends, lead_s=LEAD_S)

        frame_durations = cutpoints_to_frame_durations(cutpoints, fps=FPS)

        self.scenes = []
        cur = 0.0
        for i, seg in enumerate(segments):
            dur = float(frame_durations[i])
            vs = cur
            ve = vs + dur
            cur = ve

            self.scenes.append(Scene(
                scene_id=i+1,
                narration_text=seg["text"],
                audio_start=float(seg["start"]),
                audio_end=float(seg["end"]),
                visual_start=float(vs),
                visual_end=float(ve),
                duration=float(dur),
                status="pending",
                audio_status="pending",
            ))

        total_vid = self.scenes[-1].visual_end if self.scenes else 0.0
        diff_ms = abs(total_vid - audio_dur) * 1000.0
        _log(f"[TIMING] Escenas={len(self.scenes)} | video_total={total_vid:.6f}s | audio_total={audio_dur:.6f}s | diff={diff_ms:.1f}ms")
        self._save_state()

    # ------------------ Audio por escena (preview / persistente) ------------------
    def extract_scene_audio_clips(self) -> None:
        if not self.scenes:
            raise RuntimeError("No hay escenas. Ejecuta build_scenes_with_exact_timing primero.")
        os.makedirs(self.scene_audio_dir, exist_ok=True)

        for s in self.scenes:
            out_path = f"{self.scene_audio_dir}/scene_{s.scene_id:03d}.wav"
            cmd = (
                f"ffmpeg -y -i '{self.full_audio_path}' "
                f"-ss {s.visual_start:.6f} -to {s.visual_end:.6f} "
                f"-acodec pcm_s16le -ar 24000 '{out_path}'"
            )
            subprocess.run(cmd, shell=True, capture_output=True, text=True)
            s.audio_clip_path = out_path
            s.audio_status = "completed"

        _log(f"[AUDIO] Clips por escena extraídos: {len(self.scenes)} → {self.scene_audio_dir}")
        self._save_state()

    def rebuild_full_audio_from_scene_clips(self) -> None:
        if not self.scenes:
            return
        lst = f"{self.output_dir}/_scene_audio_concat.txt"
        with open(lst, "w", encoding="utf-8") as f:
            for s in self.scenes:
                if not s.audio_clip_path or not os.path.exists(s.audio_clip_path):
                    raise RuntimeError(f"Falta audio_clip_path para escena {s.scene_id}")
                f.write(f"file '{os.path.abspath(s.audio_clip_path)}'\n")

        tmp_out = f"{self.output_dir}/_full_narration_from_clips.wav"
        cmd = f"ffmpeg -y -f concat -safe 0 -i '{lst}' -acodec pcm_s16le -ar 24000 '{tmp_out}'"
        subprocess.run(cmd, shell=True, capture_output=True, text=True)
        os.replace(tmp_out, self.full_audio_path)
        _log("[AUDIO] full_narration.wav reconstruido desde clips por escena.")
        self._save_state()

    def regenerate_scene_audio(self, scene_id: int, new_text: str, voice: str) -> str:
        scene = next((s for s in self.scenes if s.scene_id == int(scene_id)), None)
        if not scene:
            return f"Escena {scene_id} no encontrada"

        text = (new_text or "").strip()
        if not text:
            return "Texto vacío (no se regeneró audio)."

        target = float(scene.duration)
        tmp_tts = f"{self.output_dir}/audio_chunks/_regen_scene_{scene_id:03d}.wav"

        async def _gen():
            await edge_tts.Communicate(text, voice).save(tmp_tts)

        _log(f"[AUDIO REGEN] Escena {scene_id}: generando TTS...")
        asyncio.run(_gen())

        d_new = _ffprobe_duration(tmp_tts) or 0.0
        if d_new <= 0:
            return "Fallo generando audio TTS (duración 0)."

        ratio = d_new / target if target > 0 else 1.0
        out_path = f"{self.scene_audio_dir}/scene_{scene.scene_id:03d}_regen.wav"

        if abs(d_new - target) < 0.02:
            cmd = (
                f"ffmpeg -y -i '{tmp_tts}' "
                f"-af 'atrim=0:{target:.6f},asetpts=N/SR/TB' "
                f"-acodec pcm_s16le -ar 24000 '{out_path}'"
            )
        else:
            if d_new > target:
                tempo = _build_atempo_chain(d_new / target)
                cmd = (
                    f"ffmpeg -y -i '{tmp_tts}' "
                    f"-af '{tempo},atrim=0:{target:.6f},asetpts=N/SR/TB' "
                    f"-acodec pcm_s16le -ar 24000 '{out_path}'"
                )
            else:
                cmd = (
                    f"ffmpeg -y -i '{tmp_tts}' "
                    f"-af 'apad=pad_dur=10,atrim=0:{target:.6f},asetpts=N/SR/TB' "
                    f"-acodec pcm_s16le -ar 24000 '{out_path}'"
                )

        subprocess.run(cmd, shell=True, capture_output=True, text=True)
        try:
            os.remove(tmp_tts)
        except Exception:
            pass

        scene.narration_text = text
        scene.audio_clip_path = out_path
        scene.audio_status = "completed"
        self.rebuild_full_audio_from_scene_clips()
        self._save_state()

        d_out = _ffprobe_duration(out_path) or target
        _log(f"[AUDIO REGEN] Escena {scene_id} lista. dur_out={d_out:.6f}s target={target:.6f}s")
        return f"Audio de escena {scene_id} regenerado y normalizado a {target:.2f}s. Full audio recompuesto."

    # ------------------ Prompts de imagen ------------------
    def generate_prompts_with_context(self) -> None:
        if not self.scenes:
            raise RuntimeError("No hay escenas. Ejecuta Paso 1 (WhisperX + timing) primero.")

        t0 = time.time()
        _log(f"[FASE 5] Generando prompts para {len(self.scenes)} escenas (con contexto completo del guion)...")
        running_prompts: List[str] = []
        system_prompt = build_image_system_prompt()

        for i, scene in enumerate(self.scenes):
            scene_id = scene.scene_id
            duration = scene.duration
            preview_text = scene.narration_text[:70].strip()
            _log(f"   → Escena {scene_id}/{len(self.scenes)} | {duration:.2f}s | \"{preview_text}...\"")

            user_content = build_image_prompt_request(
                full_script=self.script_text,
                segment_text=scene.narration_text,
                duration=duration,
                scene_index=scene_id,
                total_scenes=len(self.scenes),
                previous_prompts=running_prompts[-2:],
            )

            raw = self._call_xkiro_with_fallback(
                system_instruction=system_prompt,
                user_content=user_content,
                temperature=0.6,
                max_output_tokens=700,
            )
            image_prompt = sanitize_image_prompt(raw)
            running_prompts.append(image_prompt)

            scene.image_prompt = image_prompt
            scene.status = "pending"

            _log(f"     Prompt generado ({_word_count(image_prompt)} palabras): \"{image_prompt[:90]}...\"")

        self._save_state()
        self.prompts_preview_path = self.save_prompts_preview_file()
        elapsed = time.time() - t0
        avg_len = sum(_word_count(s.image_prompt) for s in self.scenes) / max(1, len(self.scenes))
        _log(f"   Prompts completados: {len(self.scenes)} | longitud media: {avg_len:.0f} palabras | tiempo: {elapsed:.1f}s")
        _log(f"   Archivo de prompts: {self.prompts_preview_path}")

    def save_prompts_preview_file(self) -> str:
        path = f"{self.output_dir}/image_prompts_{self.theme_slug}.txt"
        with open(path, "w", encoding="utf-8") as f:
            for s in self.scenes:
                f.write(f"[Escena {s.scene_id} | VIS {s.visual_start:.2f}-{s.visual_end:.2f} | dur {s.duration:.2f}s]\n")
                f.write(f"Narración: {s.narration_text}\n")
                f.write(f"Prompt: {s.image_prompt}\n\n")
        return path

    # ------------------ sd-cli runner ------------------
    def _run_sdcli_tracked(self, cmd: List[str]):
        out_f = tempfile.TemporaryFile(mode="w+")
        err_f = tempfile.TemporaryFile(mode="w+")
        t0 = time.time()
        p = subprocess.Popen(cmd, stdout=out_f, stderr=err_f, text=True)
        peak_rss = 0
        last_report = t0
        while p.poll() is None:
            if psutil is not None:
                try:
                    peak_rss = max(peak_rss, psutil.Process(p.pid).memory_info().rss)
                except Exception:
                    pass
            now = time.time()
            if now - last_report >= 30:
                cur_gb = peak_rss / 1024**3
                avail = f" | RAM libre sistema: {psutil.virtual_memory().available/1024**3:.2f} GB" if psutil else ""
                _log(f"     sd-cli trabajando... {now - t0:.0f}s | RSS pico: {cur_gb:.2f} GB{avail}")
                last_report = now
            time.sleep(0.5)
        out_f.seek(0); err_f.seek(0)
        stdout, stderr = out_f.read(), err_f.read()
        out_f.close(); err_f.close()
        return p.returncode, stdout, stderr, peak_rss / 1024**3, time.time() - t0

    # ------------------ Generación de imágenes (Pollinations) ------------------
    def _generate_one_image_pollinations(self, prompt: str, negative_prompt: str,
                                         out_path: str, model: str,
                                         width: int = 1024, height: int = 576,
                                         max_retries: int = 3) -> bool:
        BASE = "https://image.pollinations.ai/prompt"

        clean_prompt = (prompt or "").strip()
        if len(clean_prompt) > 900:
            clean_prompt = clean_prompt[:900]
            _log("   Prompt truncado a 900 chars para evitar URL overflow")

        encoded_prompt = urllib.parse.quote(clean_prompt, safe="")

        for attempt in range(1, max_retries + 1):
            try:
                seed = random.randint(0, 999999)
                url = (
                    f"{BASE}/{encoded_prompt}"
                    f"?model={model}&width={width}&height={height}&seed={seed}"
                    f"&nologo=true&referrer=videoautomation.local"
                )
                _log(f"   Pollinations intento {attempt}/{max_retries} | modelo: {model} | url_len: {len(url)}")
                response = requests.get(url, timeout=120)

                if response.status_code == 200 and len(response.content) > 5000:
                    with open(out_path, "wb") as f:
                        f.write(response.content)
                    _log(f"   OK — imagen guardada: {out_path} ({len(response.content)//1024}KB)")
                    return True

                _log(f"   HTTP {response.status_code} | size {len(response.content)} — reintento 5s")
                time.sleep(5)

            except Exception as e:
                _log(f"   Error Pollinations: {e}")
                time.sleep(5)

        return False

    def _generate_image_pollinations_with_fallback(self, prompt: str, out_path: str,
                                                  width: int = 1376, height: int = 768) -> bool:
        ok = self._generate_one_image_pollinations(
            prompt, STYLE_GUIDE["negative_prompt"], out_path,
            model=self.pollinations_model, width=width, height=height,
        )
        if ok:
            return True

        if self.pollinations_model != POLLINATIONS_FALLBACK_MODEL:
            ok = self._generate_one_image_pollinations(
                prompt, STYLE_GUIDE["negative_prompt"], out_path,
                model=POLLINATIONS_FALLBACK_MODEL, width=width, height=height,
            )
        return ok

    def _generate_one_image_qwen_local(self, prompt: str, out_path: str,
                                       width: int = 1376, height: int = 768,
                                       max_retries: int = 2) -> bool:
        if not self.qwen_local_cfg:
            raise RuntimeError("Qwen local no está configurado. Selecciona preset en la UI.")
        if not os.path.exists(SDCLI_PATH):
            raise RuntimeError(f"sd-cli no existe en {SDCLI_PATH}. Revisa Celda 1.")

        diffusion = self.qwen_local_cfg["diffusion_path"]
        llm = self.qwen_local_cfg["llm_path"]
        vae = self.qwen_local_cfg["vae_path"]
        steps = int(self.qwen_local_cfg["steps"])
        cfg = float(self.qwen_local_cfg["cfg"])

        enable_mmap = bool(self.qwen_local_cfg.get("enable_mmap", True))
        enable_vae_tiling = bool(self.qwen_local_cfg.get("enable_vae_tiling", True))
        cpu_lowram_params_backend = self.qwen_local_cfg.get("cpu_lowram_params_backend", None)
        offload_on_gpu = bool(self.qwen_local_cfg.get("offload_to_cpu_on_gpu", False))

        use_gpu = (SDCLI_BACKEND == "cuda") and bool(SDCLI_GPU_AVAILABLE)

        optional = [("--diffusion-fa", None)]
        if enable_mmap:
            optional.append(("--mmap", None))
        if enable_vae_tiling:
            optional.append(("--vae-tiling", None))
        optional.append(("--threads", str(int(SDCLI_THREADS))))
        if use_gpu and offload_on_gpu:
            optional.append(("--offload-to-cpu", None))
        if (not use_gpu) and cpu_lowram_params_backend:
            optional.append(("--params-backend", cpu_lowram_params_backend))

        optional_args, skipped = [], []
        for flag, value in optional:
            if self._sdcli_supports(flag):
                optional_args.append(flag)
                if value is not None:
                    optional_args.append(value)
            else:
                skipped.append(flag)
        if skipped:
            _log(f"   Flags no soportados por este sd-cli (omitidos): {skipped}")

        for attempt in range(1, max_retries + 1):
            try:
                seed = random.randint(0, 999999999)
                base_args = [
                    SDCLI_PATH,
                    "--diffusion-model", diffusion,
                    "--vae", vae,
                    "--llm", llm,
                    "-p", prompt,
                    "-o", out_path,
                    "--sampling-method", "euler",
                    "--cfg-scale", str(cfg),
                    "--steps", str(steps),
                    "--seed", str(seed),
                ] + optional_args

                _log(f"   sd-cli intento {attempt}/{max_retries} | {'GPU (CUDA)' if use_gpu else 'CPU'} | "
                     f"steps={steps} cfg={cfg} seed={seed} | {width}x{height}")

                cmd = base_args + ["-W", str(int(width)), "-H", str(int(height))]
                rc, out, err, peak_gb, dt = self._run_sdcli_tracked(cmd)
                _log(f"   sd-cli terminó en {dt:.1f}s | código={rc} | RSS pico: {peak_gb:.2f} GB")

                if rc == 0 and os.path.exists(out_path) and os.path.getsize(out_path) > 5000:
                    _log(f"   OK — imagen Qwen local guardada: {out_path} ({os.path.getsize(out_path)//1024}KB)")
                    return True

                cmd2 = base_args + ["--width", str(int(width)), "--height", str(int(height))]
                rc2, out2, err2, peak_gb2, dt2 = self._run_sdcli_tracked(cmd2)
                _log(f"   sd-cli (compat --width/--height) terminó en {dt2:.1f}s | código={rc2} | RSS pico: {peak_gb2:.2f} GB")

                if rc2 == 0 and os.path.exists(out_path) and os.path.getsize(out_path) > 5000:
                    _log(f"   OK — imagen Qwen local guardada: {out_path} ({os.path.getsize(out_path)//1024}KB)")
                    return True

                err_tail = (err or out or "")[-900:]
                err_tail2 = (err2 or out2 or "")[-900:]
                _log(f"   sd-cli fallo intento {attempt}/{max_retries}:\n{err_tail}\n---\n{err_tail2}")
                time.sleep(3)

            except Exception as e:
                _log(f"   Error sd-cli: {e}")
                time.sleep(3)

        return False

    # ------------------ Pruebas de imagen (RESTAURADO - ORIGINAL) ------------------
    def test_generate_image(self, prompt: str, model: str, width: int = 1376, height: int = 768) -> Optional[str]:
        images_dir = f"{self.output_dir}/images"
        os.makedirs(images_dir, exist_ok=True)
        out_path = f"{images_dir}/test_preview_{int(time.time())}.png"
        _log(f"[PRUEBA] Generando imagen de prueba con modelo '{model}'...")
        ok = self._generate_one_image_pollinations(
            prompt, STYLE_GUIDE["negative_prompt"], out_path, model=model, width=width, height=height,
        )
        if ok:
            _log(f"   Imagen de prueba lista: {out_path}")
            return out_path
        _log("   Fallo al generar imagen de prueba.")
        return None

    def test_generate_image_qwen_local(self, prompt: str, preset_label: str,
                                        width: int = 1376, height: int = 768) -> Optional[str]:
        images_dir = f"{self.output_dir}/images"
        os.makedirs(images_dir, exist_ok=True)
        out_path = f"{images_dir}/test_preview_qwen_{int(time.time())}.png"
        _log(f"[PRUEBA] Generando imagen de prueba con Qwen-Image-2.1 local | preset: {preset_label}")

        try:
            self.set_qwen_local_preset(preset_label)
        except Exception as e:
            _log(f"   Error preparando preset Qwen local: {e}")
            return None

        ok = self._generate_one_image_qwen_local(prompt, out_path, width=width, height=height)
        if ok:
            _log(f"   Imagen de prueba Qwen local lista: {out_path}")
            return out_path
        _log("   Fallo al generar imagen de prueba con Qwen local.")
        return None

    # ------------------ Generación masiva de imágenes ------------------
    def generate_images(self, progress_callback=None) -> None:
        t0 = time.time()
        _log(f"[FASE 6] Generando imágenes para {len(self.scenes)} escenas... (backend: {self.image_backend})")
        images_dir = f"{self.output_dir}/images"
        os.makedirs(images_dir, exist_ok=True)

        for idx, scene in enumerate(self.scenes):
            if scene.status == "completed" and scene.image_path and os.path.exists(scene.image_path):
                _log(f"   Escena {scene.scene_id}: ya completada, se omite.")
                continue

            scene.status = "generating"
            scene_t0 = time.time()
            try:
                img_path = f"{images_dir}/scene_{scene.scene_id:03d}.png"
                ok = False

                if self.image_backend == "pollinations":
                    ok = self._generate_image_pollinations_with_fallback(scene.image_prompt, img_path)
                    time.sleep(2.5)

                elif self.image_backend == "qwen_local":
                    ok = self._generate_one_image_qwen_local(scene.image_prompt, img_path)
                    if not ok:
                        _log("   Qwen-Image-2.1 local falló; usando Pollinations como respaldo.")
                        ok = self._generate_image_pollinations_with_fallback(scene.image_prompt, img_path)
                        time.sleep(2.5)

                else:
                    raise RuntimeError(f"Backend desconocido: {self.image_backend}")

                if not ok:
                    raise RuntimeError("No se pudo generar imagen (backend falló)")

                scene.image_path = img_path
                scene.status = "completed"
                scene_elapsed = time.time() - scene_t0
                _log(f"   Escena {scene.scene_id}/{len(self.scenes)} completada en {scene_elapsed:.1f}s")

            except Exception as e:
                _log(f"   Error escena {scene.scene_id}: {e}")
                scene.status = "error"

            if progress_callback:
                progress_callback(self.scenes)

            if (idx + 1) % 5 == 0:
                self._save_state()

        self._save_state()
        errors = sum(1 for s in self.scenes if s.status == "error")
        elapsed = time.time() - t0
        _log(f"   Imágenes terminadas. Errores: {errors} | tiempo total: {elapsed:.1f}s")

    # ------------------ Render final (timing exacto) ------------------
    def render_video(self) -> str:
        t0 = time.time()
        _log("[FASE 7] Renderizando video final con timing frame-accurate...")

        valid = [s for s in self.scenes if s.image_path and os.path.exists(s.image_path)]
        if not valid:
            raise RuntimeError("No hay imágenes válidas para renderizar")

        missing_clips = any((not s.audio_clip_path) or (not os.path.exists(s.audio_clip_path)) for s in self.scenes)
        if missing_clips:
            _log("[AUDIO] Clips faltantes: extrayendo clips por escena y reconstruyendo full audio...")
            self.extract_scene_audio_clips()
            self.rebuild_full_audio_from_scene_clips()

        concat_path = f"{self.output_dir}/video_concat.txt"
        with open(concat_path, "w", encoding="utf-8") as f:
            for scene in valid:
                f.write(f"file '{os.path.abspath(scene.image_path)}'\n")
                f.write(f"duration {scene.duration:.6f}\n")
            f.write(f"file '{os.path.abspath(valid[-1].image_path)}'\n")

        final_path = f"{self.output_dir}/final_video.mp4"
        cmd = (
            f"ffmpeg -y -f concat -safe 0 -i '{concat_path}' -i '{self.full_audio_path}' "
            f"-c:v libx264 -r {FPS} -pix_fmt yuv420p "
            f"-c:a aac -b:a 192k -shortest '{final_path}'"
        )
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        if result.returncode != 0:
            _log(f"   FFmpeg error:\n{(result.stderr or '')[-2000:]}")
            raise RuntimeError("Fallo en renderizado FFmpeg")

        video_dur = self._get_media_duration(final_path) or 0.0
        audio_dur = self._get_media_duration(self.full_audio_path) or 0.0
        _log(f"   Video final: {final_path} | VideoDur={video_dur:.6f}s | AudioDur={audio_dur:.6f}s | diff={abs(video_dur-audio_dur)*1000:.1f}ms")

        elapsed = time.time() - t0
        _log(f"   Render terminado en {elapsed:.1f}s")
        self._save_state()
        return final_path

    # ------------------ Regeneración de escena (imagen) - ORIGINAL ------------------
    def regenerate_scene(self, scene_id: int) -> str:
        scene = next((s for s in self.scenes if s.scene_id == scene_id), None)
        if not scene:
            return f"Escena {scene_id} no encontrada"

        try:
            idx = self.scenes.index(scene)
            previous_prompts = [s.image_prompt for s in self.scenes[max(0, idx - 2):idx]]

            user_content = build_image_prompt_request(
                full_script=self.script_text,
                segment_text=scene.narration_text,
                duration=scene.duration,
                scene_index=scene.scene_id,
                total_scenes=len(self.scenes),
                previous_prompts=previous_prompts,
            )
            raw = self._call_xkiro_with_fallback(
                system_instruction=build_image_system_prompt(),
                user_content=user_content,
                temperature=0.7,
                max_output_tokens=700,
            )
            new_prompt = sanitize_image_prompt(raw)
            scene.image_prompt = new_prompt
            _log(f"   Nuevo prompt para escena {scene_id}: \"{new_prompt[:100]}...\"")

            return self._regenerate_image_for_scene(scene)

        except Exception as e:
            traceback.print_exc()
            return f"Error regenerando escena {scene_id}: {e}"

    def regenerate_scene_with_prompt(self, scene_id: int, new_prompt: str) -> str:
        scene = next((s for s in self.scenes if s.scene_id == scene_id), None)
        if not scene:
            return f"Escena {scene_id} no encontrada"

        try:
            scene.image_prompt = sanitize_image_prompt(new_prompt)
            _log(f"   Prompt manual aplicado a escena {scene_id}: \"{scene.image_prompt[:100]}...\"")
            return self._regenerate_image_for_scene(scene)
        except Exception as e:
            traceback.print_exc()
            return f"Error regenerando escena {scene_id}: {e}"

    def _regenerate_image_for_scene(self, scene: Scene) -> str:
        new_path = f"{self.output_dir}/images/scene_{scene.scene_id:03d}_v2.png"

        if self.image_backend == "qwen_local":
            ok = self._generate_one_image_qwen_local(scene.image_prompt, new_path)
            if not ok:
                _log("   Qwen-Image-2.1 local falló; usando Pollinations como respaldo.")
                ok = self._generate_image_pollinations_with_fallback(scene.image_prompt, new_path)
        else:
            ok = self._generate_image_pollinations_with_fallback(scene.image_prompt, new_path)

        if not ok:
            scene.status = "error"
            self._save_state()
            return f"No se pudo regenerar la escena {scene.scene_id}"

        scene.image_path = new_path
        scene.status = "completed"
        self._save_state()
        return f"Escena {scene.scene_id} regenerada correctamente (imagen actualizada)."

engine = VideoAutomationEngine(XKIRO_API_KEY, XKIRO_BASE_URL)
_log("CELDA 3 COMPLETADA. Motor 'engine' listo.")

[16:50:10] CELDA 3 COMPLETADA. Motor 'engine' listo.
[16:50:12]    429 detectado en mistralai/mistral-large-2512. Esperando 30.0s...


In [17]:
# ═══════════════════════════════════════════════════════════════
# CELDA A: DEPENDENCIAS "EDITOR PRO" (Ken Burns + xfade)
# - Mantiene Proyecto 1 intacto (solo agrega deps que faltan)
# - Verifica que FFmpeg tenga: zoompan, xfade, select, showinfo
# ═══════════════════════════════════════════════════════════════

import os, sys, subprocess, textwrap

def sh(cmd, fatal=True):
    print(f"→ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (r.stderr or "") + "\n" + (r.stdout or "")
    if r.returncode != 0:
        print(out[-5000:])
        if fatal:
            raise RuntimeError(f"Fallo: {cmd}")
    return r

# Asegurar ffmpeg (en Kaggle ya lo instalas en Celda 1 del Proyecto 1, pero esto no estorba)
sh("apt-get update -y > /dev/null 2>&1", fatal=False)
sh("apt-get install -y ffmpeg > /dev/null 2>&1", fatal=False)

# Deps Python (mínimas para el Editor Pro)
# - pillow: por si luego quieres manipulación de imágenes
# - scenedetect + scipy: se quedan disponibles (aunque en la integración NO se usa el "import video + scene split")
sh("pip install -q pillow scenedetect scipy", fatal=False)

# Verificar filtros requeridos
filters = subprocess.run("ffmpeg -hide_banner -filters", shell=True, capture_output=True, text=True).stdout or ""
has_zoompan  = " zoompan " in filters
has_xfade    = " xfade " in filters
has_select   = " select " in filters
has_showinfo = " showinfo " in filters

print("\nFiltros requeridos (Editor Pro):")
print(" - zoompan :", "OK" if has_zoompan else "NO")
print(" - xfade   :", "OK" if has_xfade else "NO")
print(" - select  :", "OK" if has_select else "NO")
print(" - showinfo:", "OK" if has_showinfo else "NO")

if not (has_zoompan and has_xfade and has_select and has_showinfo):
    raise RuntimeError("Tu FFmpeg no trae filtros necesarios (zoompan/xfade/select/showinfo).")

print("\nCELDA A COMPLETADA.")

→ apt-get update -y > /dev/null 2>&1
→ apt-get install -y ffmpeg > /dev/null 2>&1
→ pip install -q pillow scenedetect scipy

Filtros requeridos (Editor Pro):
 - zoompan : OK
 - xfade   : OK
 - select  : OK
 - showinfo: OK

CELDA A COMPLETADA.


In [18]:
# ═══════════════════════════════════════════════════════════════
# CELDA B: EDITOR PRO (Ken Burns + xfade) INTEGRADO A PROYECTO 1
# - NO importa videos
# - NO divide escenas (usa engine.scenes del Proyecto 1)
# - Mantiene todo lo anterior intacto (solo añade un complemento)
# - Añade slider "MOTION_SPEED" para velocidad de zoom/pan
# ═══════════════════════════════════════════════════════════════

import os, time, json, random, math, shlex, subprocess, re
from dataclasses import dataclass, asdict
from typing import List, Dict, Optional, Tuple

# OUTPUT_DIR viene de tu Celda 2 del Proyecto 1
PRO_EDITOR_DIR = os.path.join(OUTPUT_DIR, "pro_editor")
PRO_WORK_DIR   = os.path.join(PRO_EDITOR_DIR, "work")
PRO_OUT_DIR    = os.path.join(PRO_EDITOR_DIR, "output")
PRO_PREVIEW_DIR= os.path.join(PRO_OUT_DIR, "preview")
PRO_FINAL_DIR  = os.path.join(PRO_OUT_DIR, "final")

for d in [PRO_EDITOR_DIR, PRO_WORK_DIR, PRO_OUT_DIR, PRO_PREVIEW_DIR, PRO_FINAL_DIR]:
    os.makedirs(d, exist_ok=True)

PRO_LOG_FILE = os.path.join(PRO_EDITOR_DIR, "editor_pro.log")
open(PRO_LOG_FILE, "a", encoding="utf-8").close()

def pro_log(msg: str):
    ts = time.strftime("%H:%M:%S")
    line = f"[{ts}] {msg}"
    print(line)
    try:
        with open(PRO_LOG_FILE, "a", encoding="utf-8") as f:
            f.write(line + "\n")
    except Exception:
        pass

def pro_read_log_tail(n=250) -> str:
    try:
        with open(PRO_LOG_FILE, "r", encoding="utf-8") as f:
            lines = f.read().splitlines()
        return "\n".join(lines[-n:])
    except Exception:
        return ""

def pro_ffprobe_duration(path: str) -> Optional[float]:
    try:
        cmd = ["ffprobe", "-v", "error", "-show_entries", "format=duration",
               "-of", "default=noprint_wrappers=1:nokey=1", path]
        r = subprocess.run(cmd, capture_output=True, text=True)
        return float((r.stdout or "").strip())
    except Exception:
        return None

# ───────────────────────────────────────────────────────────────
# Catálogos (idénticos al Proyecto 2)
# ───────────────────────────────────────────────────────────────
MOTION_TYPES_CATALOG = {
    "zoom_in":          "Zoom lento hacia adentro",
    "zoom_out_reverse": "Empieza en zoom y se aleja lentamente (reverse zoom)",
    "pan_right":        "Paneo a la derecha",
    "pan_left":         "Paneo a la izquierda",
}

XFADE_TRANSITIONS_CATALOG = [
    "fade", "dissolve", "wipeleft", "wiperight", "wipeup", "wipedown",
    "slideleft", "slideright", "slideup", "slidedown",
    "smoothleft", "smoothright", "smoothup", "smoothdown",
    "circleopen", "circleclose", "circlecrop", "rectcrop",
    "vertopen", "vertclose", "horzopen", "horzclose",
    "radial", "pixelize", "distance",
    "diagtl", "diagtr", "diagbl", "diagbr",
    "hlslice", "hrslice", "vuslice", "vdslice",
    "fadeblack", "fadewhite",
]

XFADE_TRANSITIONS_ES = {
    "fade":        "Fundido cruzado (fade)",
    "dissolve":    "Disolución suave",
    "wipeleft":    "Barrido hacia la izquierda",
    "wiperight":   "Barrido hacia la derecha",
    "wipeup":      "Barrido hacia arriba",
    "wipedown":    "Barrido hacia abajo",
    "slideleft":   "Deslizamiento a la izquierda",
    "slideright":  "Deslizamiento a la derecha",
    "slideup":     "Deslizamiento hacia arriba",
    "slidedown":   "Deslizamiento hacia abajo",
    "smoothleft":  "Deslizamiento suave a la izquierda",
    "smoothright": "Deslizamiento suave a la derecha",
    "smoothup":    "Deslizamiento suave hacia arriba",
    "smoothdown":  "Deslizamiento suave hacia abajo",
    "circleopen":  "Círculo que se abre",
    "circleclose": "Círculo que se cierra",
    "circlecrop":  "Recorte circular",
    "rectcrop":    "Recorte rectangular",
    "vertopen":    "Apertura vertical (cortinilla)",
    "vertclose":   "Cierre vertical (cortinilla)",
    "horzopen":    "Apertura horizontal (cortinilla)",
    "horzclose":   "Cierre horizontal (cortinilla)",
    "radial":      "Barrido radial (como reloj)",
    "pixelize":    "Pixelado (efecto mosaico)",
    "distance":    "Transición por distancia de color",
    "diagtl":      "Diagonal desde arriba-izquierda",
    "diagtr":      "Diagonal desde arriba-derecha",
    "diagbl":      "Diagonal desde abajo-izquierda",
    "diagbr":      "Diagonal desde abajo-derecha",
    "hlslice":     "Corte horizontal hacia la izquierda",
    "hrslice":     "Corte horizontal hacia la derecha",
    "vuslice":     "Corte vertical hacia arriba",
    "vdslice":     "Corte vertical hacia abajo",
    "fadeblack":   "Fundido a negro",
    "fadewhite":   "Fundido a blanco",
}

def to_es_label(ffmpeg_name: str) -> str:
    return XFADE_TRANSITIONS_ES.get(ffmpeg_name, ffmpeg_name)

def to_ffmpeg_name(es_label: str) -> str:
    for k, v in XFADE_TRANSITIONS_ES.items():
        if v == es_label:
            return k
    return es_label

XFADE_CHOICES_ES = [to_es_label(k) for k in XFADE_TRANSITIONS_CATALOG]

# Defaults (alineados con Proyecto 2)
DEFAULT_ENABLED_MOTIONS = ["zoom_in", "zoom_out_reverse", "pan_right", "pan_left"]
DEFAULT_ENABLED_TRANSITIONS = [
    "fade", "dissolve", "wipeleft", "wiperight",
    "slideleft", "slideright", "smoothleft", "smoothright",
    "circleopen", "radial", "pixelize",
]
DEFAULT_ENABLED_TRANSITIONS_ES = [to_es_label(k) for k in DEFAULT_ENABLED_TRANSITIONS]

# ───────────────────────────────────────────────────────────────
# Estructuras
# ───────────────────────────────────────────────────────────────
@dataclass
class ProEditScene:
    scene_id: int
    start: float
    end: float
    duration: float
    still_path: str
    motion: str = ""

PRO_STATE = {
    "scenes": [],             # List[ProEditScene]
    "plan_path": None,        # str
    "source_audio_path": None,# str
    "source_video_path": None,# str (opcional)
    "last_render_path": None, # str
}

def pro_assign_random_no_repeat(n: int, options: List[str], seed: int) -> List[str]:
    if not options:
        raise ValueError("Lista de opciones vacía.")
    rnd = random.Random(seed)
    out = []
    prev = None
    for _ in range(n):
        cand = [o for o in options if o != prev] or options
        choice = rnd.choice(cand)
        out.append(choice)
        prev = choice
    return out

def pro_assign_motions_to_scenes(scenes: List[ProEditScene], enabled_motions: List[str], seed: int) -> None:
    motions = pro_assign_random_no_repeat(len(scenes), enabled_motions, seed)
    for s, m in zip(scenes, motions):
        s.motion = m
    pro_log(f"[ProEditor/Motions] {motions[:12]}{'...' if len(motions)>12 else ''}")

def pro_assign_transitions(n_scenes: int, enabled_transitions: List[str], seed: int) -> List[str]:
    if n_scenes < 2:
        return []
    tr = pro_assign_random_no_repeat(n_scenes - 1, enabled_transitions, seed + 1000)
    pro_log(f"[ProEditor/Transitions] {tr[:12]}{'...' if len(tr)>12 else ''}")
    return tr

# ───────────────────────────────────────────────────────────────
# Motion expressions (con MOTION_SPEED)
#   speed=1.0 -> igual a Proyecto 2
#   speed>1.0 -> movimiento más rápido (llega antes al final)
#   speed<1.0 -> movimiento más lento (no llega completamente)
# ───────────────────────────────────────────────────────────────
def pro_build_motion_expr(motion: str, frames: int, zoom_max: float, pan_zoom: float, motion_speed: float) -> Tuple[str, str, str]:
    d_minus_1 = max(frames - 1, 1)
    sp = max(0.05, float(motion_speed))
    # t = min(on*sp, d_minus_1) para acelerar sin cambiar duración del clip
    t = f"min(on*{sp:.6f},{d_minus_1})"

    if motion == "zoom_in":
        z = f"1+({zoom_max}-1)*{t}/{d_minus_1}"
        x = "iw/2-(iw/zoom/2)"
        y = "ih/2-(ih/zoom/2)"

    elif motion == "zoom_out_reverse":
        z = f"{zoom_max}-({zoom_max}-1)*{t}/{d_minus_1}"
        x = "iw/2-(iw/zoom/2)"
        y = "ih/2-(ih/zoom/2)"

    elif motion == "pan_right":
        z = f"{pan_zoom}"
        x = f"(iw-iw/zoom)*{t}/{d_minus_1}"
        y = "ih/2-(ih/zoom/2)"

    elif motion == "pan_left":
        z = f"{pan_zoom}"
        x = f"(iw-iw/zoom)*(1-({t}/{d_minus_1}))"
        y = "ih/2-(ih/zoom/2)"

    else:
        raise ValueError(f"Movimiento desconocido: {motion}")

    return z, x, y

# ───────────────────────────────────────────────────────────────
# Filtergraph con xfade SIN desincronizar:
# - Mantiene longitud total ≈ sum(duraciones originales)
# - Ajusta offsets: offset = (cumulative - transition_duration)
# - Y extiende inputs desde la 2da escena: dur_i = d_i + transition_duration
# ───────────────────────────────────────────────────────────────
def pro_build_ffmpeg_filtergraph_from_stills(
    scenes: List[ProEditScene],
    transitions: List[str],
    transition_duration: float,
    out_w: int, out_h: int, fps: int,
    zoom_max: float, pan_zoom: float,
    motion_speed: float,
    quality_scale: float,
) -> Tuple[List[str], str, float, str]:

    W = max(2, int(out_w * quality_scale) // 2 * 2)
    H = max(2, int(out_h * quality_scale) // 2 * 2)
    canvas_w, canvas_h = W * 2, H * 2

    nominal_total = sum(s.duration for s in scenes)

    inputs  = []
    filters = []
    labels  = []

    # Duración por input: 1ra escena: d1, escenas>=2: d_i + td
    in_durs = []
    if len(scenes) == 1:
        in_durs = [float(scenes[0].duration)]
    else:
        in_durs = [float(scenes[0].duration)] + [float(s.duration) + float(transition_duration) for s in scenes[1:]]

    for i, (s, in_d) in enumerate(zip(scenes, in_durs)):
        if not os.path.exists(s.still_path):
            raise RuntimeError(f"Still no existe: {s.still_path}")

        frames = max(2, round(in_d * fps))
        z, x, y = pro_build_motion_expr(s.motion, frames, zoom_max, pan_zoom, motion_speed)

        inputs += ["-loop", "1", "-t", f"{in_d:.6f}", "-i", s.still_path]

        chain = (
            f"[{i}:v]"
            f"scale={canvas_w}:{canvas_h}:force_original_aspect_ratio=decrease,"
            f"pad={canvas_w}:{canvas_h}:(ow-iw)/2:(oh-ih)/2:color=black,"
            f"setsar=1,"
            f"zoompan=z='{z}':x='{x}':y='{y}':d={frames}:s={W}x{H}:fps={fps},"
            f"format=yuv420p,settb=AVTB[v{i}]"
        )
        filters.append(chain)
        labels.append(f"v{i}")

    # Encadenar xfade con offsets corregidos (para no sumar duración extra)
    current    = labels[0]
    cumulative = 0.0

    for i in range(len(scenes) - 1):
        cumulative += scenes[i].duration
        out_label = f"x{i}"

        offset = cumulative - float(transition_duration)
        if offset < 0:
            offset = 0.0

        filters.append(
            f"[{current}][{labels[i+1]}]"
            f"xfade=transition={transitions[i]}:duration={transition_duration:.6f}:"
            f"offset={offset:.6f}[{out_label}]"
        )
        current = out_label

    filter_complex = ";".join(filters)
    final_label    = current
    return inputs, filter_complex, nominal_total, final_label

def pro_run_ffmpeg_progress(cmd: List[str], nominal_total: float) -> None:
    pro_log("FFmpeg cmd:\n" + " ".join(shlex.quote(c) for c in cmd))
    t0 = time.time()
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

    last_pct = -1
    for line in proc.stdout:
        line = line.rstrip("\n")
        try:
            with open(PRO_LOG_FILE, "a", encoding="utf-8") as f:
                f.write("[FFMPEG] " + line + "\n")
        except Exception:
            pass

        if line.startswith("out_time_ms="):
            try:
                ms  = int(line.split("=", 1)[1].strip())
                sec = ms / 1_000_000.0
                pct = min(100.0, (sec / nominal_total) * 100.0) if nominal_total > 0 else 0.0
                if pct >= last_pct + 5:
                    pro_log(f"[ProEditor/Render] {pct:.1f}% ({sec:.1f}s / {nominal_total:.1f}s)")
                    last_pct = pct
            except Exception:
                pass

    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f"FFmpeg falló (rc={rc}). Revisa editor_pro.log.")
    pro_log(f"[ProEditor/Render] OK en {time.time()-t0:.1f}s")

def pro_render_from_project1_stills(
    scenes: List[ProEditScene],
    audio_path: Optional[str],
    enabled_motions: List[str],
    enabled_transitions_ffmpeg: List[str],
    transition_duration: float,
    zoom_max: float,
    pan_zoom: float,
    motion_speed: float,
    out_w: int, out_h: int, fps: int,
    seed: int,
    output_path: str,
    preview: bool,
    quality_scale: float,
) -> str:
    if not scenes:
        raise ValueError("No hay escenas (Pro Editor).")

    # Validación: duraciones mínimas vs transición
    min_dur = min(float(s.duration) for s in scenes) if scenes else 0.0
    td = float(transition_duration)

    # Clamp automático para evitar offsets negativos o escenas demasiado cortas
    if len(scenes) >= 2:
        safe_td = max(0.0, min(td, max(0.05, min_dur * 0.90)))
        if abs(safe_td - td) > 1e-9:
            pro_log(f"[ProEditor] Ajuste automático transition_duration: {td:.3f}s → {safe_td:.3f}s (min_scene={min_dur:.3f}s)")
        td = safe_td
    else:
        td = 0.0

    # cuantizar transición a grid fps
    td = round(td * fps) / fps

    pro_assign_motions_to_scenes(scenes, enabled_motions, seed)
    transitions = pro_assign_transitions(len(scenes), enabled_transitions_ffmpeg, seed) if len(scenes) >= 2 else []

    inputs, filter_complex, nominal_total, final_label = pro_build_ffmpeg_filtergraph_from_stills(
        scenes=scenes,
        transitions=transitions,
        transition_duration=td,
        out_w=out_w, out_h=out_h, fps=fps,
        zoom_max=float(zoom_max), pan_zoom=float(pan_zoom),
        motion_speed=float(motion_speed),
        quality_scale=float(quality_scale),
    )

    cmd = ["ffmpeg", "-y", "-hide_banner", "-loglevel", "info"] + inputs

    # Audio (del Proyecto 1): engine.full_audio_path (wav)
    if audio_path and os.path.exists(audio_path):
        cmd += ["-i", audio_path]

    cmd += ["-filter_complex", filter_complex, "-map", f"[{final_label}]"]

    if audio_path and os.path.exists(audio_path):
        cmd += ["-map", f"{len(scenes)}:a?"]

    cmd += ["-progress", "pipe:1", "-nostats"]

    preset = "veryfast" if preview else "medium"
    crf    = "23" if preview else "20"
    cmd += ["-c:v", "libx264", "-preset", preset, "-crf", crf, "-pix_fmt", "yuv420p", "-r", str(int(fps))]

    if audio_path and os.path.exists(audio_path):
        cmd += ["-c:a", "aac", "-b:a", "192k", "-shortest"]

    cmd += [output_path]

    pro_log(f"[ProEditor/RenderStart] preview={preview} | out={output_path}")
    pro_run_ffmpeg_progress(cmd, nominal_total)
    pro_log(f"[ProEditor/RenderDone] {output_path}")
    return output_path

def pro_build_plan_from_engine(engine_obj) -> Tuple[List[ProEditScene], str]:
    """
    Crea escenas del Editor Pro desde engine.scenes (Proyecto 1).
    Usa: image_path + visual_start/end + duration.
    """
    if not getattr(engine_obj, "scenes", None):
        raise RuntimeError("engine.scenes vacío. Ejecuta primero el flujo del Proyecto 1 (Paso 1 y generación de imágenes).")

    scenes: List[ProEditScene] = []
    for s in engine_obj.scenes:
        if not s.image_path or (not os.path.exists(s.image_path)):
            raise RuntimeError(f"Falta imagen para escena {s.scene_id}. Genera imágenes primero (Paso 2 imágenes).")
        scenes.append(ProEditScene(
            scene_id=int(s.scene_id),
            start=float(getattr(s, "visual_start", 0.0)),
            end=float(getattr(s, "visual_end", 0.0)),
            duration=float(getattr(s, "duration", 0.0)),
            still_path=str(s.image_path),
            motion="",
        ))

    plan_path = os.path.join(PRO_WORK_DIR, "edit_plan_from_project1.json")
    with open(plan_path, "w", encoding="utf-8") as f:
        json.dump([asdict(x) for x in scenes], f, ensure_ascii=False, indent=2)

    PRO_STATE["scenes"] = scenes
    PRO_STATE["plan_path"] = plan_path
    PRO_STATE["source_audio_path"] = getattr(engine_obj, "full_audio_path", None)

    pro_log(f"[ProEditor/Plan] escenas={len(scenes)} | plan={plan_path}")
    return scenes, plan_path

print("CELDA B COMPLETADA. Editor Pro backend listo.")
print("Log:", PRO_LOG_FILE)

CELDA B COMPLETADA. Editor Pro backend listo.
Log: /kaggle/working/VideoAutomationProject/pro_editor/editor_pro.log


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELDA 4: INTERFAZ GRADIO COMPLETA
# - TODAS las pestañas del Proyecto 1 intactas (sin tocar nada).
# - El Editor Pro (Ken Burns + xfade) se integra en la pestaña 1
#   ("Generar Nuevo Video"), justo después del botón de render final,
#   como un bloque colapsable "Edición Pro (Ken Burns + Transiciones)".
# - NO se crea pestaña nueva.
# - Se añade slider de velocidad zoom/pan.
# ═══════════════════════════════════════════════════════════════

import gradio as gr
import pandas as pd
import os
import time
import traceback
import re
from typing import List

# ──────────────────────────────────────────────────────────────
# Helpers Proyecto 1 (idénticos al original)
# ──────────────────────────────────────────────────────────────
def read_log_tail(n_lines: int = 250) -> str:
    path = os.environ.get("PROJECT_LOG_FILE", f"{OUTPUT_DIR}/run.log")
    if not os.path.exists(path):
        return "(sin log aún)"
    try:
        with open(path, "r", encoding="utf-8") as f:
            lines = f.read().splitlines()
        return "\n".join(lines[-n_lines:])
    except Exception as e:
        return f"(error leyendo log: {e})"

def scenes_to_dataframe(scenes: List[Scene]) -> pd.DataFrame:
    if not scenes:
        return pd.DataFrame(columns=["ID", "VIS Inicio", "VIS Fin", "Duración",
                                     "ASR Inicio", "ASR Fin", "Texto",
                                     "Estado Img", "Estado Aud"])
    return pd.DataFrame([{
        "ID": s.scene_id,
        "VIS Inicio": round(s.visual_start, 3),
        "VIS Fin": round(s.visual_end, 3),
        "Duración": round(s.duration, 3),
        "ASR Inicio": round(s.audio_start, 3),
        "ASR Fin": round(s.audio_end, 3),
        "Texto": (s.narration_text[:80] + ("..." if len(s.narration_text) > 80 else "")),
        "Estado Img": s.status,
        "Estado Aud": s.audio_status,
    } for s in scenes])

def scenes_to_dataframe_full(scenes: List[Scene]) -> pd.DataFrame:
    if not scenes:
        return pd.DataFrame(columns=["ID", "Duración", "Narración", "Prompt de Imagen"])
    return pd.DataFrame([{
        "ID": s.scene_id,
        "Duración": round(s.duration, 3),
        "Narración": s.narration_text,
        "Prompt de Imagen": s.image_prompt,
    } for s in scenes])

def scenes_to_gallery_items(scenes: List[Scene]):
    items = []
    for s in scenes:
        if s.image_path and os.path.exists(s.image_path):
            items.append((s.image_path, f"Escena {s.scene_id} | {s.duration:.2f}s"))
    return items

# ──────────────────────────────────────────────────────────────
# Funciones de flujo Proyecto 1 (idénticas al original)
# ──────────────────────────────────────────────────────────────
def run_stage1(theme, duration_min, voice_label, text_model_label):
    voice = resolve_voice_id(voice_label, theme)
    text_model_id = XKIRO_MODEL_OPTIONS.get(text_model_label, DEFAULT_XKIRO_TEXT_MODEL_ID)
    engine.set_text_model(text_model_id)
    try:
        set_video_language_from_title(theme)
        print("Idioma detectado:", VIDEO_LANG_CODE, VIDEO_LANG_NAME)
        
        yield "Generando guion...", pd.DataFrame(), None, None, pd.DataFrame(), read_log_tail()
        script = engine.generate_script(theme, duration_min)

        yield "Generando audio (TTS) + normalización a frame-grid...", pd.DataFrame(), None, None, pd.DataFrame(), read_log_tail()
        audio_path = engine.generate_audio(script, voice)

        yield "Transcribiendo con WhisperX (align)...", pd.DataFrame(), None, audio_path, pd.DataFrame(), read_log_tail()
        segments = engine.transcribe_and_align(audio_path)

        yield "Construyendo escenas con timing exacto (silences + frames)...", pd.DataFrame(), None, audio_path, pd.DataFrame(), read_log_tail()
        engine.build_scenes_with_exact_timing(segments)

        yield "Extrayendo audio por escena + recomponiendo full audio...", pd.DataFrame(), None, audio_path, scenes_to_dataframe(engine.scenes), read_log_tail()
        engine.extract_scene_audio_clips()
        engine.rebuild_full_audio_from_scene_clips()

        yield "Generando prompts de imagen (contexto completo)...", pd.DataFrame(), None, engine.full_audio_path, scenes_to_dataframe(engine.scenes), read_log_tail()
        engine.generate_prompts_with_context()

        yield ("PASO 1 COMPLETADO — revisa Audio/Prompts antes de generar imágenes",
               scenes_to_dataframe_full(engine.scenes),
               engine.prompts_preview_path,
               engine.full_audio_path,
               scenes_to_dataframe(engine.scenes),
               read_log_tail())
    except Exception as e:
        traceback.print_exc()
        yield f"ERROR: {e}", scenes_to_dataframe_full(engine.scenes), None, None, scenes_to_dataframe(engine.scenes), read_log_tail()

def run_stage2(image_backend_label, pollinations_model_label, qwen_preset_label):
    image_backend = IMAGE_BACKENDS.get(image_backend_label, "pollinations")
    pollinations_key = POLLINATIONS_MODELS.get(pollinations_model_label, "flux")
    engine.set_image_backend(image_backend)
    engine.set_image_model(pollinations_key)
    try:
        if not engine.scenes:
            yield "ERROR: primero ejecuta el PASO 1 (guion + prompts).", None, pd.DataFrame()
            return
        if image_backend == "qwen_local":
            yield "Preparando modelos Qwen-Image-2.1 local...", None, scenes_to_dataframe(engine.scenes)
            engine.set_qwen_local_preset(qwen_preset_label)
        yield "Generando imágenes...", None, scenes_to_dataframe(engine.scenes)
        engine.generate_images()
        yield "Renderizando video final...", None, scenes_to_dataframe(engine.scenes)
        video_path = engine.render_video()
        yield "COMPLETADO", video_path, scenes_to_dataframe(engine.scenes)
    except Exception as e:
        traceback.print_exc()
        yield f"ERROR: {e}", None, scenes_to_dataframe(engine.scenes)

def run_stage2_images_only(image_backend_label, pollinations_model_label, qwen_preset_label):
    image_backend = IMAGE_BACKENDS.get(image_backend_label, "pollinations")
    pollinations_key = POLLINATIONS_MODELS.get(pollinations_model_label, "flux")
    engine.set_image_backend(image_backend)
    engine.set_image_model(pollinations_key)
    try:
        if not engine.scenes:
            return "ERROR: primero ejecuta el PASO 1.", scenes_to_dataframe(engine.scenes), scenes_to_gallery_items(engine.scenes), read_log_tail()
        if image_backend == "qwen_local":
            engine.set_qwen_local_preset(qwen_preset_label)
        engine.generate_images()
        return "Imágenes generadas. Revisa en Línea de Tiempo y renderiza cuando estés listo.", scenes_to_dataframe(engine.scenes), scenes_to_gallery_items(engine.scenes), read_log_tail()
    except Exception as e:
        traceback.print_exc()
        return f"ERROR: {e}", scenes_to_dataframe(engine.scenes), scenes_to_gallery_items(engine.scenes), read_log_tail()

def do_final_render():
    try:
        path = engine.render_video()
        return f"Render final OK: {path}", path, scenes_to_dataframe(engine.scenes), read_log_tail()
    except Exception as e:
        traceback.print_exc()
        return f"ERROR render: {e}", None, scenes_to_dataframe(engine.scenes), read_log_tail()

def run_full_auto(theme, duration_min, voice_label, text_model_label,
                  image_backend_label, pollinations_model_label, qwen_preset_label):
    last_status = "Iniciando..."
    for status, df_full, prompts_file, audio_out, timing_df, log_out in run_stage1(
            theme, duration_min, voice_label, text_model_label):
        last_status = status
        yield status, None, pd.DataFrame(), df_full, prompts_file
    if last_status.startswith("ERROR"):
        return
    for status, video_path, df_short in run_stage2(
            image_backend_label, pollinations_model_label, qwen_preset_label):
        yield status, video_path, df_short, scenes_to_dataframe_full(engine.scenes), engine.prompts_preview_path

# ──────────────────────────────────────────────────────────────
# Vista Previa & Prueba de Modelos (idéntico al original)
# ──────────────────────────────────────────────────────────────
def load_scene_prompt(scene_id):
    try:
        scene = next((s for s in engine.scenes if s.scene_id == int(scene_id)), None)
        if not scene:
            return "", "", "Escena no encontrada"
        return scene.narration_text, scene.image_prompt, f"Escena {scene_id} cargada"
    except Exception as e:
        return "", "", f"Error: {e}"

def toggle_test_backend_visibility(backend_choice):
    is_qwen = (backend_choice == "Qwen-Image-2.1 Local (sd-cli)")
    return gr.update(visible=not is_qwen), gr.update(visible=is_qwen)

def generate_test_image_unified(prompt_text, backend_choice, pollinations_model_label, qwen_preset_label):
    try:
        if not prompt_text or not prompt_text.strip():
            return None, "Prompt vacío. Carga una escena primero o escribe un prompt."
        prompt_text = prompt_text.strip()
        if backend_choice == "Qwen-Image-2.1 Local (sd-cli)":
            t0 = time.time()
            path = engine.test_generate_image_qwen_local(prompt_text, qwen_preset_label)
            dt = time.time() - t0
            if not path:
                return None, f"Fallo con Qwen local | preset: {qwen_preset_label}. Revisa consola."
            return path, f"Qwen local | preset: {qwen_preset_label} | tiempo: {dt:.1f}s"
        else:
            model_key = POLLINATIONS_MODELS.get(pollinations_model_label, "flux")
            path = engine.test_generate_image(prompt_text, model_key)
            if not path:
                return None, f"Fallo al generar con '{model_key}'."
            return path, f"Imagen generada con modelo '{model_key}'."
    except Exception as e:
        traceback.print_exc()
        return None, f"Error: {e}"

# ──────────────────────────────────────────────────────────────
# Editor de Escenas (idéntico al original)
# ──────────────────────────────────────────────────────────────
def regenerate_scene_ai(scene_id):
    try:
        msg = engine.regenerate_scene(int(scene_id))
        return msg, scenes_to_dataframe(engine.scenes)
    except Exception as e:
        return f"Error: {e}", scenes_to_dataframe(engine.scenes)

def regenerate_scene_manual(scene_id, edited_prompt):
    try:
        msg = engine.regenerate_scene_with_prompt(int(scene_id), edited_prompt)
        return msg, scenes_to_dataframe(engine.scenes)
    except Exception as e:
        return f"Error: {e}", scenes_to_dataframe(engine.scenes)

# ──────────────────────────────────────────────────────────────
# Línea de Tiempo (idéntico al original)
# ──────────────────────────────────────────────────────────────
def load_scene_full(scene_id):
    try:
        scene_id = int(scene_id)
        scene = next((s for s in engine.scenes if s.scene_id == scene_id), None)
        if not scene:
            return "", "", None, None, "Escena no encontrada", scenes_to_dataframe(engine.scenes), read_log_tail()
        aud = scene.audio_clip_path if scene.audio_clip_path and os.path.exists(scene.audio_clip_path) else None
        img = scene.image_path if scene.image_path and os.path.exists(scene.image_path) else None
        status = (f"Escena {scene.scene_id} | VIS {scene.visual_start:.2f}-{scene.visual_end:.2f} "
                  f"({scene.duration:.2f}s) | Img={scene.status} | Aud={scene.audio_status}")
        return scene.narration_text, scene.image_prompt, aud, img, status, scenes_to_dataframe(engine.scenes), read_log_tail()
    except Exception as e:
        return "", "", None, None, f"Error: {e}", scenes_to_dataframe(engine.scenes), read_log_tail()

def on_gallery_select(evt: gr.SelectData):
    try:
        items = scenes_to_gallery_items(engine.scenes)
        if evt.index is None or evt.index < 0 or evt.index >= len(items):
            return "", "", None, None, "Selección inválida", scenes_to_dataframe(engine.scenes), read_log_tail()
        label = items[evt.index][1]
        m = re.search(r"Escena\s+(\d+)", label)
        if not m:
            return "", "", None, None, "No se pudo resolver escena", scenes_to_dataframe(engine.scenes), read_log_tail()
        return load_scene_full(int(m.group(1)))
    except Exception as e:
        return "", "", None, None, f"Error: {e}", scenes_to_dataframe(engine.scenes), read_log_tail()

def regenerate_audio_ui(scene_id, edited_text, voice_label):
    try:
        voice = VOICE_OPTIONS.get(voice_label, "es-ES-AlvaroNeural")
        msg = engine.regenerate_scene_audio(int(scene_id), edited_text, voice)
        return msg, scenes_to_dataframe(engine.scenes), read_log_tail()
    except Exception as e:
        traceback.print_exc()
        return f"Error: {e}", scenes_to_dataframe(engine.scenes), read_log_tail()

def regenerate_scene_ai_timeline(scene_id):
    try:
        msg = engine.regenerate_scene(int(scene_id))
        return msg, scenes_to_dataframe(engine.scenes), scenes_to_gallery_items(engine.scenes), read_log_tail()
    except Exception as e:
        traceback.print_exc()
        return f"Error: {e}", scenes_to_dataframe(engine.scenes), scenes_to_gallery_items(engine.scenes), read_log_tail()

def regenerate_scene_manual_timeline(scene_id, edited_prompt):
    try:
        msg = engine.regenerate_scene_with_prompt(int(scene_id), edited_prompt)
        return msg, scenes_to_dataframe(engine.scenes), scenes_to_gallery_items(engine.scenes), read_log_tail()
    except Exception as e:
        traceback.print_exc()
        return f"Error: {e}", scenes_to_dataframe(engine.scenes), scenes_to_gallery_items(engine.scenes), read_log_tail()

# ──────────────────────────────────────────────────────────────
# Persistencia Kaggle (idéntico al original)
# ──────────────────────────────────────────────────────────────
def publish_dataset_ui(dataset_slug, title, update_existing):
    try:
        if not dataset_slug or not dataset_slug.strip():
            return "Escribe un slug (ej: qwen-image-2-1-models)."
        return publish_models_as_kaggle_dataset(
            dataset_slug.strip(), title.strip() if title else None, bool(update_existing))
    except Exception as e:
        traceback.print_exc()
        return f"Error: {e}"

# ──────────────────────────────────────────────────────────────
# Editor Pro helpers UI
# ──────────────────────────────────────────────────────────────
def pro_scenes_to_df_ui(scenes: List[ProEditScene]) -> pd.DataFrame:
    if not scenes:
        return pd.DataFrame(columns=["ID", "Start", "End", "Duración", "Still", "Motion"])
    return pd.DataFrame([{
        "ID": s.scene_id,
        "Start": round(s.start, 3),
        "End": round(s.end, 3),
        "Duración": round(s.duration, 3),
        "Still": os.path.basename(s.still_path),
        "Motion": s.motion or "(sin asignar)",
    } for s in scenes])

def pro_ui_sync_from_project1():
    try:
        scenes, plan_path = pro_build_plan_from_engine(engine)
        return (
            f"✅ Plan Pro creado: {len(scenes)} escenas | {plan_path}",
            plan_path,
            pro_scenes_to_df_ui(scenes),
            pro_read_log_tail(),
        )
    except Exception as e:
        traceback.print_exc()
        return f"❌ ERROR: {e}", None, pd.DataFrame(), pro_read_log_tail()

def pro_ui_render(
    mode_label,
    qscale,
    seed,
    transition_dur,
    zoom_max,
    pan_zoom,
    motion_speed,
    enabled_motions,
    enabled_transitions_es,
    out_w, out_h, out_fps,
    use_project1_audio,
):
    try:
        scenes = PRO_STATE.get("scenes") or []
        if not scenes:
            return (
                "❌ ERROR: primero pulsa 'Cargar escenas del Proyecto 1'. "
                "También necesitas haber generado las imágenes (Paso 2 imágenes).",
                None, pd.DataFrame(), pro_read_log_tail(),
            )
        if not enabled_motions:
            return "❌ ERROR: selecciona al menos 1 movimiento.", None, pro_scenes_to_df_ui(scenes), pro_read_log_tail()
        if len(scenes) >= 2 and not enabled_transitions_es:
            return "❌ ERROR: selecciona al menos 1 transición.", None, pro_scenes_to_df_ui(scenes), pro_read_log_tail()

        enabled_transitions_ffmpeg = [to_ffmpeg_name(lbl) for lbl in (enabled_transitions_es or [])]

        preview  = (mode_label == "Preview (rápido)")
        out_dir  = PRO_PREVIEW_DIR if preview else PRO_FINAL_DIR
        out_path = os.path.join(out_dir, f"{'preview' if preview else 'final'}_{int(time.time())}.mp4")

        audio_path = None
        if bool(use_project1_audio):
            audio_path = getattr(engine, "full_audio_path", None)
            if not audio_path or not os.path.exists(audio_path):
                return (
                    "❌ ERROR: no se encontró engine.full_audio_path. "
                    "Ejecuta PASO 1 primero.",
                    None, pro_scenes_to_df_ui(scenes), pro_read_log_tail(),
                )

        path = pro_render_from_project1_stills(
            scenes=scenes,
            audio_path=audio_path,
            enabled_motions=list(enabled_motions),
            enabled_transitions_ffmpeg=enabled_transitions_ffmpeg,
            transition_duration=float(transition_dur),
            zoom_max=float(zoom_max),
            pan_zoom=float(pan_zoom),
            motion_speed=float(motion_speed),
            out_w=int(out_w), out_h=int(out_h), fps=int(out_fps),
            seed=int(seed),
            output_path=out_path,
            preview=preview,
            quality_scale=float(qscale) if preview else 1.0,
        )
        PRO_STATE["last_render_path"] = path
        label = "✅ Preview Pro listo" if preview else "✅ Render Pro FINAL listo"
        return f"{label}: {path}", path, pro_scenes_to_df_ui(scenes), pro_read_log_tail()

    except Exception as e:
        traceback.print_exc()
        return f"❌ ERROR: {e}", None, pd.DataFrame(), pro_read_log_tail()

def pro_ui_export_all():
    files = []
    p = PRO_STATE.get("plan_path")
    if p and os.path.exists(p):
        files.append(p)
    if os.path.exists(PRO_LOG_FILE):
        files.append(PRO_LOG_FILE)
    main_log = os.environ.get("PROJECT_LOG_FILE", f"{OUTPUT_DIR}/run.log")
    if main_log and os.path.exists(main_log):
        files.append(main_log)
    return files

# ══════════════════════════════════════════════════════════════
# UI GRADIO COMPLETA
# ══════════════════════════════════════════════════════════════
with gr.Blocks(title="Automatización de Videos - Garabatos 2D") as demo:

    gr.Markdown("# Sistema de Automatización — Garabatos 2D (xKiro + Pollinations/Qwen Local)")
    gr.Markdown(
        f"**Plataforma:** `{PLATFORM}` | **sd-cli backend:** `{SDCLI_BACKEND}` | "
        f"**GPU runtime:** `{SDCLI_GPU_AVAILABLE}` | **FPS:** `{FPS}` | **LEAD:** `{LEAD_MS}ms`"
    )

    with gr.Tabs():

        # ══════════════════════════════════════════════════════
        # TAB 1 — Generar Nuevo Video  (ORIGINAL + Editor Pro al final)
        # ══════════════════════════════════════════════════════
        with gr.Tab("Generar Nuevo Video"):
            with gr.Row():

                # ── Columna izquierda ─────────────────────────
                with gr.Column(scale=1):

                    # ─── BLOQUE ORIGINAL ─────────────────────
                    theme_input = gr.Textbox(label="Tema del video")
                    duration_slider = gr.Slider(
                        minimum=1, maximum=15, value=8, step=0.5,
                        label="Duración objetivo (minutos)")
                    voice_dropdown = gr.Dropdown(
                        choices=list(VOICE_OPTIONS.keys()),
                        value="Álvaro (Hombre, España)",
                        label="Voz narradora")
                    text_model_dropdown = gr.Dropdown(
                        choices=list(XKIRO_MODEL_OPTIONS.keys()),
                        value=DEFAULT_XKIRO_TEXT_MODEL_LABEL,
                        label="Modelo de texto (xKiro)")
                    stage1_btn = gr.Button(
                        "PASO 1 → Guion + Audio + Transcripción + Timing + Prompts",
                        variant="secondary")

                    gr.Markdown("---")

                    image_backend_dropdown = gr.Dropdown(
                        choices=list(IMAGE_BACKENDS.keys()),
                        value=DEFAULT_IMAGE_BACKEND_LABEL,
                        label="Backend de imagen")
                    pollinations_dropdown = gr.Dropdown(
                        choices=list(POLLINATIONS_MODELS.keys()),
                        value=DEFAULT_POLLINATIONS_MODEL,
                        label="Modelo Pollinations (si usas Pollinations)")
                    qwen_preset_dropdown = gr.Dropdown(
                        choices=list(QWEN21_LOCAL_PRESETS.keys()),
                        value=DEFAULT_QWEN21_PRESET_LABEL,
                        label="Preset Qwen local (si usas Qwen local)")
                    stage2_btn = gr.Button(
                        "PASO 2 (clásico) → Generar Imágenes + Renderizar Video",
                        variant="primary")
                    stage2_images_only_btn = gr.Button(
                        "PASO 2 (nuevo) → Solo Generar Imágenes (SIN render, para supervisar)")

                    gr.Markdown("---")
                    render_final_btn = gr.Button(
                        "PASO 3 → RENDER FINAL (cuando lo apruebes)",
                        variant="primary")

                    gr.Markdown("---")
                    auto_btn = gr.Button(
                        "Ejecutar TODO Automático (Paso 1 + Paso 2 clásico)")

                    # ─── BLOQUE NUEVO: EDITOR PRO ─────────────
                    gr.Markdown("---")
                    gr.Markdown(
                        "## Edición Pro (Ken Burns + Transiciones)\n"
                        "Aplica movimiento y transiciones cinematográficas sobre las imágenes "
                        "ya generadas en el Paso 2. Pulsa primero **Cargar escenas** y luego "
                        "**Preview Pro** o **Render Pro FINAL**."
                    )

                    pro_sync_btn = gr.Button(
                        "Cargar escenas del Proyecto 1 → crear plan Pro",
                        variant="secondary")
                    pro_sync_status = gr.Textbox(
                        label="Estado carga plan Pro",
                        interactive=False, lines=2)
                    pro_plan_file = gr.File(
                        label="Plan Pro JSON (descargable)")

                    gr.Markdown("#### Modo y calidad")
                    pro_mode = gr.Radio(
                        ["Preview (rápido)", "Final (calidad completa)"],
                        value="Preview (rápido)",
                        label="Modo render Pro")
                    pro_qscale = gr.Slider(
                        0.25, 0.80, value=0.45, step=0.05,
                        label="Escala resolución (solo en Preview)")
                    pro_seed = gr.Number(
                        value=42, label="Semilla aleatoria (reproduce resultados)", precision=0)

                    gr.Markdown("#### Movimientos (Ken Burns, aleatorios por escena)")
                    pro_enabled_motions = gr.CheckboxGroup(
                        choices=list(MOTION_TYPES_CATALOG.keys()),
                        value=DEFAULT_ENABLED_MOTIONS,
                        label="Movimientos permitidos")
                    pro_zoom_max = gr.Slider(
                        1.05, 1.80, value=1.18, step=0.01,
                        label="Zoom máximo (zoom_in / zoom_out_reverse)")
                    pro_pan_zoom = gr.Slider(
                        1.05, 1.50, value=1.15, step=0.01,
                        label="Zoom fijo en paneos (pan_left / pan_right)")
                    pro_motion_speed = gr.Slider(
                        0.30, 3.00, value=1.00, step=0.05,
                        label="Velocidad zoom/pan (1.0 = normal · >1 más rápido · <1 más lento)")

                    gr.Markdown("#### Transiciones (xfade, aleatorias)")
                    pro_enabled_transitions_es = gr.CheckboxGroup(
                        choices=XFADE_CHOICES_ES,
                        value=DEFAULT_ENABLED_TRANSITIONS_ES,
                        label="Transiciones habilitadas")
                    pro_transition_dur = gr.Slider(
                        0.05, 1.50, value=0.25, step=0.05,
                        label="Duración transición (s) — se ajusta si hay escenas muy cortas")

                    gr.Markdown("#### Audio y resolución de salida")
                    pro_use_audio = gr.Checkbox(
                        value=True,
                        label="Usar audio del Proyecto 1 (engine.full_audio_path)")
                    pro_out_w = gr.Number(value=1280, label="Ancho salida (px)", precision=0)
                    pro_out_h = gr.Number(value=720,  label="Alto salida (px)",  precision=0)
                    pro_out_fps = gr.Number(value=30, label="FPS salida",        precision=0)

                    with gr.Row():
                        pro_preview_btn = gr.Button("Preview Pro", variant="secondary")
                        pro_final_btn   = gr.Button("Render Pro FINAL", variant="primary")

                    pro_export_btn  = gr.Button("Exportar plan Pro + logs")
                    pro_export_files = gr.File(
                        label="Archivos descargables (plan + logs)",
                        file_count="multiple")
                    pro_log_box = gr.Textbox(
                        label="Log Editor Pro",
                        value=pro_read_log_tail(),
                        lines=12, interactive=False)

                # ── Columna derecha ───────────────────────────
                with gr.Column(scale=2):

                    # ─── SALIDAS ORIGINALES ───────────────────
                    status_md = gr.Markdown("Esperando...")
                    prompts_table = gr.Dataframe(
                        label="Guion + Prompts de Imagen (revisa antes de generar imágenes)",
                        wrap=True)
                    prompts_file_out = gr.File(label="Descargar prompts (.txt)")
                    full_audio_out = gr.Audio(
                        label="Audio completo (full_narration.wav)",
                        type="filepath")
                    timing_table = gr.Dataframe(
                        label="Timing (VIS vs ASR) — debería ser contiguo y coherente",
                        wrap=True)
                    video_out = gr.Video(label="Preview del video final (render clásico)")
                    table_out = gr.Dataframe(label="Escenas (resumen)")
                    log_tail = gr.Textbox(
                        label="Log (tail run.log)",
                        value=read_log_tail(), lines=15, interactive=False)

                    # ─── SALIDAS EDITOR PRO ───────────────────
                    gr.Markdown("---")
                    gr.Markdown("### Resultados Editor Pro")
                    pro_render_status = gr.Textbox(
                        label="Estado render Pro",
                        interactive=False, lines=2)
                    pro_df = gr.Dataframe(
                        label="Tabla escenas Pro (motion asignado)", wrap=True)
                    pro_video_out = gr.Video(
                        label="Video Pro renderizado (Ken Burns + Transiciones)")

            # ─── Conexiones botones ORIGINALES ─────────────────
            stage1_btn.click(
                fn=run_stage1,
                inputs=[theme_input, duration_slider, voice_dropdown, text_model_dropdown],
                outputs=[status_md, prompts_table, prompts_file_out,
                         full_audio_out, timing_table, log_tail])

            stage2_btn.click(
                fn=run_stage2,
                inputs=[image_backend_dropdown, pollinations_dropdown, qwen_preset_dropdown],
                outputs=[status_md, video_out, table_out])

            stage2_images_only_btn.click(
                fn=run_stage2_images_only,
                inputs=[image_backend_dropdown, pollinations_dropdown, qwen_preset_dropdown],
                outputs=[status_md, timing_table,
                         gr.Gallery(visible=False),   # timeline_gallery placeholder
                         log_tail])

            render_final_btn.click(
                fn=do_final_render,
                outputs=[status_md, video_out, timing_table, log_tail])

            auto_btn.click(
                fn=run_full_auto,
                inputs=[theme_input, duration_slider, voice_dropdown, text_model_dropdown,
                        image_backend_dropdown, pollinations_dropdown, qwen_preset_dropdown],
                outputs=[status_md, video_out, table_out, prompts_table, prompts_file_out])

            # ─── Conexiones botones EDITOR PRO ─────────────────
            pro_sync_btn.click(
                fn=pro_ui_sync_from_project1,
                outputs=[pro_sync_status, pro_plan_file, pro_df, pro_log_box])

            pro_preview_btn.click(
                fn=pro_ui_render,
                inputs=[
                    pro_mode, pro_qscale, pro_seed,
                    pro_transition_dur, pro_zoom_max, pro_pan_zoom,
                    pro_motion_speed,
                    pro_enabled_motions, pro_enabled_transitions_es,
                    pro_out_w, pro_out_h, pro_out_fps,
                    pro_use_audio,
                ],
                outputs=[pro_render_status, pro_video_out, pro_df, pro_log_box])

            pro_final_btn.click(
                fn=pro_ui_render,
                inputs=[
                    pro_mode, pro_qscale, pro_seed,
                    pro_transition_dur, pro_zoom_max, pro_pan_zoom,
                    pro_motion_speed,
                    pro_enabled_motions, pro_enabled_transitions_es,
                    pro_out_w, pro_out_h, pro_out_fps,
                    pro_use_audio,
                ],
                outputs=[pro_render_status, pro_video_out, pro_df, pro_log_box])

            pro_export_btn.click(
                fn=pro_ui_export_all,
                outputs=[pro_export_files])

        # ══════════════════════════════════════════════════════
        # TAB 2 — Vista Previa & Prueba de Modelos (ORIGINAL, intacto)
        # ══════════════════════════════════════════════════════
        with gr.Tab("Vista Previa & Prueba de Modelos"):
            gr.Markdown("Prueba rápida: Pollinations o Qwen local (14 presets).")
            with gr.Row():
                with gr.Column():
                    preview_scene_id = gr.Number(label="ID escena (opcional)", precision=0)
                    load_scene_btn = gr.Button("Cargar Prompt de esta Escena")
                    preview_narration = gr.Textbox(
                        label="Narración", interactive=False, lines=3)
                    preview_prompt = gr.Textbox(
                        label="Prompt (editable)", lines=6)
                    test_backend_radio = gr.Radio(
                        choices=["Pollinations", "Qwen-Image-2.1 Local (sd-cli)"],
                        value="Pollinations", label="Backend a probar")
                    preview_model_dropdown = gr.Dropdown(
                        choices=list(POLLINATIONS_MODELS.keys()),
                        value=DEFAULT_POLLINATIONS_MODEL,
                        label="Modelo Pollinations", visible=True)
                    preview_qwen_preset_dropdown = gr.Dropdown(
                        choices=list(QWEN21_LOCAL_PRESETS.keys()),
                        value=DEFAULT_QWEN21_PRESET_LABEL,
                        label="Preset Qwen local", visible=False)
                    test_backend_radio.change(
                        fn=toggle_test_backend_visibility,
                        inputs=[test_backend_radio],
                        outputs=[preview_model_dropdown, preview_qwen_preset_dropdown])
                    test_btn = gr.Button("Generar Imagen de Prueba", variant="primary")
                    preview_status = gr.Textbox(label="Estado", interactive=False)
                with gr.Column():
                    test_image_out = gr.Image(label="Resultado")

            load_scene_btn.click(
                fn=load_scene_prompt,
                inputs=[preview_scene_id],
                outputs=[preview_narration, preview_prompt, preview_status])
            test_btn.click(
                fn=generate_test_image_unified,
                inputs=[preview_prompt, test_backend_radio,
                        preview_model_dropdown, preview_qwen_preset_dropdown],
                outputs=[test_image_out, preview_status])

        # ══════════════════════════════════════════════════════
        # TAB 3 — Editor de Escenas (ORIGINAL, intacto)
        # ══════════════════════════════════════════════════════
        with gr.Tab("Editor de Escenas"):
            gr.Markdown("Edita prompt y regenera escena.")
            edit_scene_id = gr.Number(label="ID escena", precision=0)
            load_edit_btn = gr.Button("Cargar Escena")
            edit_narration = gr.Textbox(
                label="Narración", interactive=False, lines=3)
            edit_prompt_box = gr.Textbox(
                label="Prompt (edítalo)", lines=6)
            with gr.Row():
                manual_regen_btn = gr.Button("Regenerar (manual)", variant="primary")
                ai_regen_btn     = gr.Button("Regenerar (IA)")
            regen_status = gr.Textbox(label="Estado", interactive=False)
            regen_table  = gr.Dataframe(label="Escenas")

            load_edit_btn.click(
                fn=load_scene_prompt,
                inputs=[edit_scene_id],
                outputs=[edit_narration, edit_prompt_box, regen_status])
            manual_regen_btn.click(
                fn=regenerate_scene_manual,
                inputs=[edit_scene_id, edit_prompt_box],
                outputs=[regen_status, regen_table])
            ai_regen_btn.click(
                fn=regenerate_scene_ai,
                inputs=[edit_scene_id],
                outputs=[regen_status, regen_table])

        # ══════════════════════════════════════════════════════
        # TAB 4 — Línea de Tiempo (Supervisión) (ORIGINAL, intacto)
        # ══════════════════════════════════════════════════════
        with gr.Tab("Línea de Tiempo (Supervisión)"):
            gr.Markdown(
                "### Supervisión total antes del render final: "
                "audio por escena + imagen por escena")
            with gr.Row():
                with gr.Column(scale=1):
                    scene_id_box = gr.Number(
                        label="ID escena", precision=0, value=1)
                    load_btn = gr.Button("Cargar escena")
                    narration_edit = gr.Textbox(
                        label="Narración (editable para regenerar audio)", lines=3)
                    prompt_edit = gr.Textbox(
                        label="Prompt de imagen (editable)", lines=6)
                    with gr.Row():
                        regen_audio_btn    = gr.Button("🔊 Regenerar audio de escena")
                        regen_img_ai_btn   = gr.Button("🎨 Regenerar imagen (IA)")
                        regen_img_manual_btn = gr.Button("🎨 Regenerar imagen (prompt manual)")
                    scene_status = gr.Textbox(label="Estado", interactive=False)
                    log_tail2 = gr.Textbox(
                        label="Log (tail run.log)",
                        value=read_log_tail(), lines=15, interactive=False)
                with gr.Column(scale=1):
                    scene_audio_player = gr.Audio(
                        label="Audio escena (preview/descarga)", type="filepath")
                    scene_image_preview = gr.Image(
                        label="Imagen escena (preview/descarga)")

            timeline_df = gr.Dataframe(
                label="Tabla timing/estado",
                value=scenes_to_dataframe(engine.scenes), wrap=True)
            timeline_gallery_main = gr.Gallery(
                label="Galería (escenas con imagen)",
                value=scenes_to_gallery_items(engine.scenes),
                columns=6, height=420)

            refresh_btn = gr.Button("Actualizar tabla/galería")
            refresh_btn.click(
                fn=lambda: (
                    scenes_to_dataframe(engine.scenes),
                    scenes_to_gallery_items(engine.scenes),
                    read_log_tail()),
                outputs=[timeline_df, timeline_gallery_main, log_tail2])

            load_btn.click(
                fn=load_scene_full,
                inputs=[scene_id_box],
                outputs=[narration_edit, prompt_edit, scene_audio_player,
                         scene_image_preview, scene_status, timeline_df, log_tail2])

            timeline_gallery_main.select(
                fn=on_gallery_select,
                outputs=[narration_edit, prompt_edit, scene_audio_player,
                         scene_image_preview, scene_status, timeline_df, log_tail2])

            regen_audio_btn.click(
                fn=regenerate_audio_ui,
                inputs=[scene_id_box, narration_edit, voice_dropdown],
                outputs=[scene_status, timeline_df, log_tail2])

            regen_img_ai_btn.click(
                fn=regenerate_scene_ai_timeline,
                inputs=[scene_id_box],
                outputs=[scene_status, timeline_df, timeline_gallery_main, log_tail2])

            regen_img_manual_btn.click(
                fn=regenerate_scene_manual_timeline,
                inputs=[scene_id_box, prompt_edit],
                outputs=[scene_status, timeline_df, timeline_gallery_main, log_tail2])

            final_render_btn2 = gr.Button(
                "🎬 RENDER FINAL (desde Línea de Tiempo)", variant="primary")
            final_video_out2 = gr.Video(label="Video final")
            final_render_btn2.click(
                fn=do_final_render,
                outputs=[scene_status, final_video_out2, timeline_df, log_tail2])

        # ══════════════════════════════════════════════════════
        # TAB 5 — Persistencia de Modelos (Kaggle) (ORIGINAL, intacto)
        # ══════════════════════════════════════════════════════
        with gr.Tab("Persistencia de Modelos (Kaggle)"):
            gr.Markdown(
                "Publica los modelos descargados como Dataset "
                "(requiere Kaggle Secrets: KAGGLE_USERNAME y KAGGLE_KEY).")
            dataset_slug_box = gr.Textbox(
                label="Slug dataset", value="qwen-image-2-1-models")
            dataset_title_box = gr.Textbox(
                label="Título dataset (opcional)",
                value="Qwen Image 2.1 - Modelos GGUF")
            dataset_update_checkbox = gr.Checkbox(
                label="Actualizar dataset existente", value=True)
            publish_btn = gr.Button(
                "Publicar/Actualizar Dataset", variant="primary")
            publish_status = gr.Textbox(
                label="Resultado", interactive=False, lines=6)

            publish_btn.click(
                fn=publish_dataset_ui,
                inputs=[dataset_slug_box, dataset_title_box, dataset_update_checkbox],
                outputs=[publish_status])

demo.launch(share=True, debug=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://8aef67c1e4c86aab40.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[16:53:30] Cambiando modelo de texto xKiro: qwen/qwen3.8-max:free → qwen/qwen3.5-flash:free
Idioma detectado: es ESPAÑOL
[16:53:30] [FASE 2] Generando guion: '¿Qué sucedía realmente cuando los Homo Sapiens y los Neandertales se encontraban?' (7.5 min)
[16:53:30]    Objetivo: ~1125 palabras | modelo principal: qwen/qwen3.5-flash:free
[16:54:08]    Guion generado: 1262 palabras en 37.4s
[16:54:08]    Preview: "Imagina que caminas por una calle concurrida de tu ciudad un martes por la tarde. Ves a alguien con ropa extraña, quizás..."
[16:54:08] [FASE 3] Generando audio con voz: es-ES-AlvaroNeural
[16:54:08]    Guion dividido en 18 fragmentos
[16:54:10]    Fragmento 1/18 generado (482 caracteres)
[16:54:16]    Fragmento 2/18 generado (449 caracteres)
[16:54:18]    Fragmento 3/18 generado (486 caracteres)
[16:54:22]    Fragmento 4/18 generado (495 caracteres)
[16:54:27]    Fragmento 5/18 generado (397 caracteres)
[16:54:29]    Fragmento 6/18 generado (483 caracteres)
[16:54:33]    Fragmento

Using cache found in /root/.cache/torch/hub/snakers4_silero-vad_master


2026-09-28 16:55:31 - whisperx.asr - INFO - Detected language: es (1.00) in first 30s of audio


/tmp/ipykernel_1148/1902930576.py:659: UserWarning: batch_size (16) is more than half the total VAD chunks (18). Streams will have very few segments; consider reducing --batch_size for better context.
  result = whisper_model.transcribe(audio, batch_size=16)


[16:55:59]    Transcripción cruda: 18 segmentos detectados
[16:56:07]    Alineación completada: 106 segmentos finales | duración media/segmento: 4.78s | tiempo: 48.1s
[16:56:08] [TIMING] Duración audio (ffprobe): 508.100000s | FPS=30 | LEAD=120ms
[16:56:08] [silencedetect] silence_end detectados: 161 (noise=-35dB, d=0.15)
[16:56:08] [TIMING] Escenas=106 | video_total=508.100000s | audio_total=508.100000s | diff=0.0ms
[16:56:20] [AUDIO] Clips por escena extraídos: 106 → /kaggle/working/VideoAutomationProject/scene_audio_clips
[16:56:21] [AUDIO] full_narration.wav reconstruido desde clips por escena.
[16:56:21] [FASE 5] Generando prompts para 106 escenas (con contexto completo del guion)...
[16:56:21]    → Escena 1/106 | 4.93s | "Imagina que caminas por una calle concurrida de tu ciudad un martes po..."
[16:56:29]      Prompt generado (185 palabras): "Detailed 2D cartoon animation style, simple flat textures, thick and well-defined black li..."
[16:56:29]    → Escena 2/106 | 7.47s | "Ves

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELDA 5: LIMPIEZA DE MODELOS QWEN (liberar espacio Kaggle ~20GB)
# - Lista modelos en /kaggle/working/.../local_models
# - Muestra tamaño por archivo y total
# - Permite borrar selectivamente (con confirmación)
# - Mantiene los que quieras usar con regularidad
# ═══════════════════════════════════════════════════════════════

import os
import shutil
import time
from typing import List, Tuple

import gradio as gr
import pandas as pd

# Reutiliza rutas del proyecto (Celda 2 debe haberse ejecutado)
try:
    _ = OUTPUT_DIR
    _ = LOCAL_MODELS_DIR
except NameError:
    OUTPUT_DIR = "/kaggle/working/VideoAutomationProject"
    LOCAL_MODELS_DIR = f"{OUTPUT_DIR}/local_models"

os.makedirs(LOCAL_MODELS_DIR, exist_ok=True)

# También limpiamos staging de publicación de datasets (si existe y pesa)
STAGING_DIR = f"{OUTPUT_DIR}/dataset_publish_staging"
OTHER_HEAVY_DIRS = [
    LOCAL_MODELS_DIR,
    STAGING_DIR,
    f"{OUTPUT_DIR}/images",           # opcional: previews enormes
    f"{OUTPUT_DIR}/scene_audio_clips",
]

MODEL_EXTS = (".gguf", ".safetensors", ".bin", ".pt", ".pth", ".ckpt", ".part")

def _bytes_to_gb(n: int) -> float:
    return n / (1024 ** 3)

def _human_size(n: int) -> str:
    if n < 1024:
        return f"{n} B"
    if n < 1024 ** 2:
        return f"{n/1024:.1f} KB"
    if n < 1024 ** 3:
        return f"{n/1024**2:.2f} MB"
    return f"{n/1024**3:.2f} GB"

def _disk_usage_working() -> Tuple[float, float, float]:
    """Uso de /kaggle/working (total, usado, libre) en GB."""
    try:
        usage = shutil.disk_usage("/kaggle/working")
        return (
            _bytes_to_gb(usage.total),
            _bytes_to_gb(usage.used),
            _bytes_to_gb(usage.free),
        )
    except Exception:
        return 0.0, 0.0, 0.0

def _scan_model_files(root: str) -> List[dict]:
    items = []
    if not os.path.isdir(root):
        return items
    for dirpath, _, filenames in os.walk(root):
        for fn in filenames:
            # incluir .part (descargas a medias que ocupan espacio)
            if not fn.lower().endswith(MODEL_EXTS) and not fn.endswith(".part"):
                continue
            path = os.path.join(dirpath, fn)
            try:
                size = os.path.getsize(path)
            except OSError:
                continue
            rel = os.path.relpath(path, root)
            items.append({
                "name": fn,
                "rel_path": rel,
                "full_path": path,
                "size_bytes": size,
                "size_human": _human_size(size),
                "folder": root,
            })
    # más grandes primero
    items.sort(key=lambda x: x["size_bytes"], reverse=True)
    return items

def scan_all_models() -> List[dict]:
    all_items = []
    # principal: local_models
    all_items.extend(_scan_model_files(LOCAL_MODELS_DIR))
    # staging de dataset (copias)
    if os.path.isdir(STAGING_DIR):
        all_items.extend(_scan_model_files(STAGING_DIR))
    # dedupe por full_path
    seen = set()
    unique = []
    for it in all_items:
        if it["full_path"] in seen:
            continue
        seen.add(it["full_path"])
        unique.append(it)
    unique.sort(key=lambda x: x["size_bytes"], reverse=True)
    return unique

def models_to_dataframe(items: List[dict]) -> pd.DataFrame:
    if not items:
        return pd.DataFrame(columns=["#", "Archivo", "Tamaño", "Ubicación", "Ruta relativa"])
    rows = []
    for i, it in enumerate(items, 1):
        loc = "local_models" if it["folder"].rstrip("/").endswith("local_models") else os.path.basename(it["folder"])
        rows.append({
            "#": i,
            "Archivo": it["name"],
            "Tamaño": it["size_human"],
            "Ubicación": loc,
            "Ruta relativa": it["rel_path"],
        })
    return pd.DataFrame(rows)

def build_status_md(items: List[dict]) -> str:
    total_b = sum(it["size_bytes"] for it in items)
    total_gb, used_gb, free_gb = _disk_usage_working()
    lines = [
        "### Estado de almacenamiento",
        f"- **Modelos/archivos listados:** {len(items)}",
        f"- **Peso total de lo listado:** **{_human_size(total_b)}** ({_bytes_to_gb(total_b):.2f} GB)",
        f"- **/kaggle/working** → total ≈ {total_gb:.1f} GB | usado ≈ {used_gb:.1f} GB | libre ≈ {free_gb:.1f} GB",
        f"- Carpeta principal: `{LOCAL_MODELS_DIR}`",
    ]
    if free_gb < 2.0 and free_gb > 0:
        lines.append("\n⚠️ **Poco espacio libre** (< 2 GB). Borra modelos que no uses.")
    if used_gb >= 18.0:
        lines.append("\n⚠️ **Cerca del límite de persistencia (~20 GB)**. Prioriza borrar `.gguf` grandes.")
    return "\n".join(lines)

def refresh_ui():
    items = scan_all_models()
    choices = [f"{it['name']}  |  {it['size_human']}  |  {it['rel_path']}" for it in items]
    # mapa label -> path (se guarda en state vía closure en handlers)
    df = models_to_dataframe(items)
    status = build_status_md(items)
    return status, df, gr.update(choices=choices, value=[]), items

def delete_selected(selected_labels: List[str], items_state: List[dict], confirm_text: str):
    if not selected_labels:
        return "No hay nada seleccionado.", *refresh_ui()[0:3], items_state if items_state is not None else scan_all_models()

    if (confirm_text or "").strip().upper() != "BORRAR":
        return (
            "⚠️ Para confirmar, escribe exactamente: **BORRAR** en la casilla de confirmación.",
            build_status_md(items_state or scan_all_models()),
            models_to_dataframe(items_state or scan_all_models()),
            gr.update(),
            items_state or scan_all_models(),
        )

    # reconstruir mapa desde estado actual
    items = items_state or scan_all_models()
    label_to_path = {
        f"{it['name']}  |  {it['size_human']}  |  {it['rel_path']}": it["full_path"]
        for it in items
    }

    deleted = []
    failed = []
    freed = 0

    for label in selected_labels:
        path = label_to_path.get(label)
        if not path:
            # intento por nombre solo
            name = label.split("  |  ")[0].strip()
            path = next((it["full_path"] for it in items if it["name"] == name), None)
        if not path or not os.path.exists(path):
            failed.append(f"{label} (no encontrado)")
            continue
        try:
            sz = os.path.getsize(path)
            os.remove(path)
            freed += sz
            deleted.append(f"{os.path.basename(path)} ({_human_size(sz)})")
        except Exception as e:
            failed.append(f"{path}: {e}")

    # limpia directorios vacíos residuales en local_models/staging
    for root in (LOCAL_MODELS_DIR, STAGING_DIR):
        if not os.path.isdir(root):
            continue
        for dirpath, dirnames, filenames in os.walk(root, topdown=False):
            if dirpath == root:
                continue
            try:
                if not os.listdir(dirpath):
                    os.rmdir(dirpath)
            except Exception:
                pass

    msg_lines = [f"✅ Liberado: **{_human_size(freed)}**"]
    if deleted:
        msg_lines.append("**Borrados:**")
        msg_lines.extend([f"- {d}" for d in deleted])
    if failed:
        msg_lines.append("**Fallos:**")
        msg_lines.extend([f"- {f}" for f in failed])

    status, df, choices_update, new_items = refresh_ui()
    return "\n".join(msg_lines), status, df, choices_update, new_items

def delete_all_part_files(items_state: List[dict], confirm_text: str):
    """Borra solo descargas a medias (*.part) — suele recuperar mucho espacio basura."""
    if (confirm_text or "").strip().upper() != "BORRAR":
        return "⚠️ Escribe **BORRAR** para confirmar la limpieza de archivos .part.", build_status_md(items_state or []), models_to_dataframe(items_state or []), gr.update(), items_state

    items = items_state or scan_all_models()
    selected = []
    for it in items:
        if it["name"].endswith(".part") or it["full_path"].endswith(".part"):
            selected.append(f"{it['name']}  |  {it['size_human']}  |  {it['rel_path']}")
    if not selected:
        return "No hay archivos .part para borrar.", *refresh_ui()[0:3], scan_all_models()
    return delete_selected(selected, items, "BORRAR")

def keep_only_selected(selected_labels: List[str], items_state: List[dict], confirm_text: str):
    """
    Estrategia útil: seleccionas los que QUIERES CONSERVAR y borra el resto.
    """
    if (confirm_text or "").strip().upper() != "BORRAR":
        return (
            "⚠️ Para confirmar 'dejar solo los seleccionados', escribe **BORRAR**.",
            build_status_md(items_state or []),
            models_to_dataframe(items_state or []),
            gr.update(),
            items_state,
        )
    items = items_state or scan_all_models()
    if not items:
        return "No hay modelos.", *refresh_ui()[0:3], []

    keep_names = set()
    for label in (selected_labels or []):
        keep_names.add(label.split("  |  ")[0].strip())

    if not keep_names:
        return (
            "⚠️ No seleccionaste ninguno para conservar. "
            "Si quieres borrar TODO, usa el botón de borrar seleccionados eligiendo todos, "
            "o selecciona los que sí quieres guardar.",
            build_status_md(items),
            models_to_dataframe(items),
            gr.update(),
            items,
        )

    to_delete_labels = []
    for it in items:
        if it["name"] not in keep_names:
            to_delete_labels.append(f"{it['name']}  |  {it['size_human']}  |  {it['rel_path']}")

    if not to_delete_labels:
        return "Nada que borrar: ya solo están los seleccionados.", *refresh_ui()[0:3], scan_all_models()

    return delete_selected(to_delete_labels, items, "BORRAR")

def quick_space_report() -> str:
    items = scan_all_models()
    total_b = sum(it["size_bytes"] for it in items)
    total_gb, used_gb, free_gb = _disk_usage_working()

    # top 10
    top = items[:10]
    lines = [
        f"[{time.strftime('%H:%M:%S')}] Reporte rápido",
        f"working: usado={used_gb:.2f}GB libre={free_gb:.2f}GB total≈{total_gb:.2f}GB",
        f"modelos listados: {len(items)} | peso={_human_size(total_b)}",
        "",
        "TOP archivos:",
    ]
    for it in top:
        lines.append(f"  {_human_size(it['size_bytes']):>10}  {it['full_path']}")
    if not top:
        lines.append("  (vacío)")
    return "\n".join(lines)

# ---------------------------------------------------------------
# UI
# ---------------------------------------------------------------
with gr.Blocks(title="Limpieza de modelos Qwen") as demo_cleanup:
    gr.Markdown("# 🧹 Limpieza de modelos Qwen / espacio Kaggle")
    gr.Markdown(
        "Usa esto para borrar GGUF/VAE que ya no necesitas y liberar el límite de **~20GB** de persistencia.\n\n"
        f"**Carpeta de modelos:** `{LOCAL_MODELS_DIR}`"
    )

    items_state = gr.State([])

    status_md = gr.Markdown("Cargando...")
    df_out = gr.Dataframe(label="Modelos encontrados", wrap=True)

    with gr.Row():
        refresh_btn = gr.Button("🔄 Actualizar lista", variant="secondary")
        report_btn = gr.Button("📊 Reporte rápido (consola/UI)")

    model_select = gr.CheckboxGroup(label="Selecciona archivos", choices=[])

    confirm_box = gr.Textbox(
        label="Confirmación de seguridad",
        placeholder="Escribe BORRAR para confirmar cualquier borrado",
        lines=1,
    )

    with gr.Row():
        del_btn = gr.Button("🗑️ Borrar seleccionados", variant="stop")
        del_part_btn = gr.Button("🧽 Borrar solo archivos .part (descargas rotas)")
        keep_btn = gr.Button("✅ Dejar SOLO los seleccionados (borra el resto)", variant="primary")

    result_md = gr.Markdown("")
    report_box = gr.Textbox(label="Reporte", lines=12, interactive=False)

    gr.Markdown(
        """
### Recomendación práctica
1. Mira la tabla (ordenada por tamaño).
2. Quédate solo con **1 preset** que uses de verdad, por ejemplo:
   - 1× Diffusion (`Q2_K` / `Q4_K` / etc.)
   - 1× Encoder (`Q2_K` o `Q4_K_M`)
   - 1× VAE (`qwen_image_2.1_vae_bf16.safetensors`)
3. Selecciona los que quieres **conservar** → botón **Dejar SOLO los seleccionados**.
4. O selecciona basura → **Borrar seleccionados**.

**Importante:** escribir `BORRAR` es obligatorio para cualquier eliminación.
        """
    )

    def _on_refresh():
        status, df, choices_update, items = refresh_ui()
        return status, df, choices_update, items, ""

    refresh_btn.click(
        fn=_on_refresh,
        outputs=[status_md, df_out, model_select, items_state, result_md],
    )

    report_btn.click(fn=quick_space_report, outputs=[report_box])

    del_btn.click(
        fn=delete_selected,
        inputs=[model_select, items_state, confirm_box],
        outputs=[result_md, status_md, df_out, model_select, items_state],
    )

    del_part_btn.click(
        fn=delete_all_part_files,
        inputs=[items_state, confirm_box],
        outputs=[result_md, status_md, df_out, model_select, items_state],
    )

    keep_btn.click(
        fn=keep_only_selected,
        inputs=[model_select, items_state, confirm_box],
        outputs=[result_md, status_md, df_out, model_select, items_state],
    )

    # carga inicial
    demo_cleanup.load(
        fn=_on_refresh,
        outputs=[status_md, df_out, model_select, items_state, result_md],
    )

demo_cleanup.launch(share=True, debug=True)